In [ ]:
import json
import re
from pathlib import Path
from typing import Dict, List, Optional


OUTPUT_DIR = "outputs"


def find_input_file() -> Path:
    """
    Automatically find the SpeakX knowledge bank markdown file.
    """
    exact_names = [
        "speakx_knowledge_bank.md",
        "speakx knowledge bank.md",
        "knowledge_bank.md",
    ]

    search_roots = [
        Path("."),
        Path("/mnt/data"),
        Path("/content"),
    ]

    # 1. Try exact filenames first
    for root in search_roots:
        if root.exists():
            for name in exact_names:
                candidate = root / name
                if candidate.exists():
                    return candidate

    # 2. Try recursive search for close matches
    patterns = [
        "*speakx*knowledge*bank*.md",
        "*knowledge*bank*.md",
        "*.md",
    ]

    for root in search_roots:
        if root.exists():
            for pattern in patterns:
                matches = list(root.rglob(pattern))
                if matches:
                    # Prefer files with "speakx" in name
                    matches.sort(key=lambda p: ("speakx" not in p.name.lower(), len(str(p))))
                    return matches[0]

    raise FileNotFoundError(
        "Could not find the input markdown file.\n"
        "Put your file in the current folder or /mnt/data, and name it like:\n"
        "speakx_knowledge_bank.md"
    )


def read_text(path: Path) -> str:
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")
    return path.read_text(encoding="utf-8")


def clean(text: Optional[str]) -> Optional[str]:
    if text is None:
        return None
    text = re.sub(r"\s+", " ", text).strip()
    return text or None


def extract_section(text: str, heading: str) -> str:
    pattern = rf"^## {re.escape(heading)}\s*$([\s\S]*?)(?=^## |\Z)"
    match = re.search(pattern, text, flags=re.MULTILINE)
    return match.group(1).strip() if match else ""


def extract_subsection(text: str, heading: str) -> str:
    pattern = rf"^### {re.escape(heading)}\s*$([\s\S]*?)(?=^### |\Z)"
    match = re.search(pattern, text, flags=re.MULTILINE)
    return match.group(1).strip() if match else ""


def parse_bullet_list(block: str) -> List[str]:
    items = []
    for line in block.splitlines():
        line = line.strip()
        if line.startswith("- "):
            items.append(clean(line[2:]))
    return [x for x in items if x]


def parse_north_star(md: str) -> Dict:
    section = extract_section(md, "North Star Metric")

    primary_metric = None
    m = re.search(r"\*\*Primary Metric\*\*:\s*(.+)", section)
    if m:
        primary_metric = clean(m.group(1))

    justification = []
    before_supporting = section.split("**Supporting Metrics:**")[0]
    for line in before_supporting.splitlines():
        line = line.strip()
        if line.startswith("- "):
            justification.append(clean(line[2:]))

    supporting_metrics = []
    after_supporting = ""
    if "**Supporting Metrics:**" in section:
        after_supporting = section.split("**Supporting Metrics:**", 1)[1]

    definition_map = {
        "Trial → Paid conversion rate": "Percentage of trial users who become paying subscribers.",
        "W1 Retention post-conversion": "Percentage of converted users retained through the first week after conversion.",
        "Monthly Retention (D30+)": "Percentage of users retained after 30+ days.",
        "Average practice sessions per active week": "Average number of practice sessions completed by an active learner in a week.",
    }

    reason_map = {
        "Trial → Paid conversion rate": "Measures monetization efficiency from the trial experience.",
        "W1 Retention post-conversion": "Shows whether the early paid experience creates stickiness.",
        "Monthly Retention (D30+)": "Tracks sustained product value over time.",
        "Average practice sessions per active week": "Measures consistency and habit strength.",
    }

    for line in after_supporting.splitlines():
        line = line.strip()
        if not line.startswith("- "):
            continue

        item = line[2:].strip()
        metric_match = re.match(r"(.+?)\s*\(Target:\s*(.+?)\)", item)

        if metric_match:
            metric_name = clean(metric_match.group(1))
            target = clean(metric_match.group(2))
        else:
            metric_name = clean(item)
            target = None

        supporting_metrics.append({
            "metric": metric_name,
            "definition": definition_map.get(metric_name),
            "target": target,
            "reason": reason_map.get(metric_name),
        })

    measurable_proxy = {
        "name": primary_metric,
        "definition": "Count of monthly active learners who maintain a 7+ day practice streak.",
        "formula": "Monthly active learners with streak >= 7 days",
        "time_window": "Monthly",
        "reason": "The knowledge bank explicitly defines this as the primary metric for sustained engagement and habit formation.",
    }

    return {
        "north_star_metric": primary_metric,
        "justification": justification,
        "measurable_proxy": measurable_proxy,
        "supporting_metrics": supporting_metrics,
        "confidence_note": "High confidence because the knowledge bank explicitly names the primary metric and supporting metrics.",
    }


def infer_lifecycle_stage(feature: Optional[str], expected_outcome: Optional[str], success_metric: Optional[str]) -> Optional[str]:
    text = " ".join([feature or "", expected_outcome or "", success_metric or ""]).lower()

    if any(word in text for word in ["trial", "paid", "conversion", "subscribe"]):
        return "conversion"
    if any(word in text for word in ["retention", "streak", "weekly active", "monthly retention", "active week"]):
        return "retention"
    if any(word in text for word in ["adoption", "feature use", "session completion", "practice session"]):
        return "activation"
    if any(word in text for word in ["re-engage", "come back", "return"]):
        return "re-engagement"

    return "activation"


def parse_feature_goal_map(md: str) -> Dict:
    section = extract_section(md, "Feature → Goal → Outcome Mapping")
    lines = [line.rstrip() for line in section.splitlines() if line.strip()]

    rows = []
    in_table = False

    for line in lines:
        if line.startswith("| Feature |"):
            in_table = True
            continue
        if in_table and line.startswith("|---"):
            continue
        if in_table and line.startswith("|"):
            parts = [p.strip() for p in line.strip("|").split("|")]
            if len(parts) >= 4:
                feature, user_goal, expected_outcome, success_metric = parts[:4]
                feature = clean(feature)
                user_goal = clean(user_goal)
                expected_outcome = clean(expected_outcome)
                success_metric = clean(success_metric)

                rows.append({
                    "feature": feature,
                    "lifecycle_stage": infer_lifecycle_stage(feature, expected_outcome, success_metric),
                    "user_goal": user_goal,
                    "expected_outcome": expected_outcome,
                    "success_metric": success_metric,
                    "evidence": [
                        f"{feature} → {user_goal} → {expected_outcome} → {success_metric}"
                    ],
                })

    return {"feature_goal_outcome_map": rows}


def parse_allowed_tones(md: str) -> List[Dict]:
    section = extract_section(md, "Ethical Communication Guidelines")
    allowed_block = extract_subsection(section, "Allowed Tones & Themes")

    chunks = re.split(r"(?=\*\*.+?\*\*\s*✓)", allowed_block)
    tones = []

    desc_map = {
        "Empowerment": "Affirms learner growth, agency, and confidence-building.",
        "Accomplishment": "Highlights milestones, progress, and achievements.",
        "Encouragement": "Supports continued practice with positive reinforcement.",
        "Social Influence": "Uses peers or broader community as positive motivation.",
        "Scarcity (Positive)": "Creates urgency without fear or manipulation.",
        "Ownership": "Frames progress, balance, or pathway as belonging to the learner.",
    }

    allowed_for_map = {
        "Empowerment": ["confidence building", "habit reinforcement", "progress updates"],
        "Accomplishment": ["milestone messages", "streak celebrations", "progress notifications"],
        "Encouragement": ["daily reminders", "re-engagement", "consistency nudges"],
        "Social Influence": ["leaderboards", "community features", "peer-based motivation"],
        "Scarcity (Positive)": ["trial conversion", "limited-time opportunities", "live sessions"],
        "Ownership": ["coins", "progress reports", "learning path reminders"],
    }

    disallowed_for_map = {
        "Empowerment": ["shame-based reminders"],
        "Accomplishment": ["false achievement claims"],
        "Encouragement": ["pressure tactics"],
        "Social Influence": ["harmful comparison", "humiliation"],
        "Scarcity (Positive)": ["fear-heavy loss framing", "deceptive urgency"],
        "Ownership": ["guilt-heavy possession framing"],
    }

    for chunk in chunks:
        chunk = chunk.strip()
        if not chunk.startswith("**"):
            continue

        header_match = re.match(r"\*\*(.+?)\*\*\s*✓", chunk)
        if not header_match:
            continue

        tone = clean(header_match.group(1))
        bullets = parse_bullet_list(chunk)

        tones.append({
            "tone": tone,
            "description": desc_map.get(tone),
            "allowed_for": allowed_for_map.get(tone, []),
            "disallowed_for": disallowed_for_map.get(tone, []),
            "evidence": bullets[:3],
        })

    return tones


def parse_disallowed_tones(md: str) -> List[Dict]:
    section = extract_section(md, "Ethical Communication Guidelines")
    disallowed_block = extract_subsection(section, "Disallowed Tones & Themes")

    if not disallowed_block:
        return []

    chunks = re.split(r"(?=\*\*.+?\*\*)", disallowed_block)
    results = []

    for chunk in chunks:
        chunk = chunk.strip()
        if not chunk.startswith("**"):
            continue

        header_match = re.match(r"\*\*(.+?)\*\*", chunk)
        if not header_match:
            continue

        tone = clean(header_match.group(1))
        bullets = parse_bullet_list(chunk)
        reason = bullets[0] if bullets else None

        results.append({
            "tone": tone,
            "reason": reason,
            "evidence": bullets[:3],
        })

    return results


def infer_stages_from_hook(title: Optional[str]) -> List[str]:
    if not title:
        return []

    lower = title.lower()

    if "accomplishment" in lower:
        return ["activation", "retention"]
    if "social influence" in lower:
        return ["activation", "retention"]
    if "scarcity" in lower:
        return ["conversion", "re-engagement"]
    if "unpredictability" in lower:
        return ["activation", "retention"]
    if "loss" in lower:
        return ["re-engagement", "retention"]
    if "ownership" in lower:
        return ["retention", "conversion"]
    if "epic meaning" in lower:
        return ["awareness", "activation"]
    if "creativity" in lower:
        return ["activation", "retention"]

    return []


def parse_hooks(md: str) -> List[Dict]:
    section = extract_section(md, "Octolysis Framework Hooks")

    pattern = r"^### (.+?)\s*$([\s\S]*?)(?=^### |\Z)"
    matches = re.finditer(pattern, section, flags=re.MULTILINE)

    hooks = []
    for match in matches:
        title = clean(match.group(1))
        body = match.group(2)

        app = re.search(r"\*\*Scenario Application\*\*:\s*(.+)", body)
        msg = re.search(r"\*\*Message Hook\*\*:\s*(.+)", body)
        goal = re.search(r"\*\*Goal Alignment\*\*:\s*(.+)", body)

        scenario_text = clean(app.group(1)) if app else None
        message_hook = clean(msg.group(1)) if msg else None
        goal_alignment = clean(goal.group(1)) if goal else None

        best_features = []
        if scenario_text:
            best_features = [x.strip() for x in scenario_text.split(",") if x.strip()]

        hooks.append({
            "hook": message_hook,
            "type": title,
            "description": goal_alignment,
            "best_for_stages": infer_stages_from_hook(title),
            "best_for_features": best_features,
            "risk_notes": "Use ethically and avoid manipulative framing.",
            "evidence": [x for x in [scenario_text, message_hook, goal_alignment] if x],
        })

    return hooks


def build_outputs(md: str) -> Dict[str, Dict]:
    return {
        "company_north_star": parse_north_star(md),
        "feature_goal_map": parse_feature_goal_map(md),
        "allowed_tone_hook_matrix": {
            "allowed_tones": parse_allowed_tones(md),
            "disallowed_tones": parse_disallowed_tones(md),
            "hook_taxonomy": parse_hooks(md),
        },
    }


def save_outputs(outputs: Dict[str, Dict], output_dir: Path) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)

    file_map = {
        "company_north_star.json": outputs["company_north_star"],
        "feature_goal_map.json": outputs["feature_goal_map"],
        "allowed_tone_hook_matrix.json": outputs["allowed_tone_hook_matrix"],
    }

    for filename, payload in file_map.items():
        path = output_dir / filename
        path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
        print(f"saved: {path}")


def main() -> None:
    input_file = find_input_file()
    output_dir = Path(OUTPUT_DIR)

    print(f"Found input file: {input_file}")
    md = read_text(input_file)
    outputs = build_outputs(md)
    save_outputs(outputs, output_dir)

    print("\nDone. Generated files:")
    for p in output_dir.glob("*.json"):
        print(p)


main()

Found input file: speakx_knowledge_bank.md
saved: outputs/company_north_star.json
saved: outputs/feature_goal_map.json
saved: outputs/allowed_tone_hook_matrix.json

Done. Generated files:
outputs/allowed_tone_hook_matrix.json
outputs/company_north_star.json
outputs/feature_goal_map.json


In [ ]:
import pandas as pd
import numpy as np
import warnings
from google.colab import userdata
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, label_binarize
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import classification_report, roc_auc_score

warnings.filterwarnings("ignore")


# =========================================================
# 1. CONFIG
# =========================================================
INPUT_FILE  = "cleaned_synthetic_behavioral_dataset_2000.xlsx"
SHEET_NAME  = "Synthetic_2000"
HEADER_ROW  = 3

# Whether to include non-behavior columns like age_band / region
INCLUDE_PROFILE_COLUMNS = True

# Whether to include lifecycle_stage in the clustering step
# (churn risk is computed separately — this only affects K-Means features)
INCLUDE_LIFECYCLE_STAGE = False

RANDOM_STATE = 42

# ── Churn risk label ordering (lowest → highest risk) ─────────────────────
# These are the four label strings that will appear in the output CSV.
# Cutoffs between them are computed from the data (percentile-based), NOT hardcoded.
CHURN_RISK_LABELS = ["low", "medium", "high", "critical"]


# =========================================================
# 2. LOAD DATA
# =========================================================
def load_data(file_path, sheet_name=None, header_row=0):
    if file_path.lower().endswith(".csv"):
        df = pd.read_csv(file_path)
    else:
        df = pd.read_excel(file_path, sheet_name=sheet_name, header=header_row)

    df = df.dropna(how="all").dropna(axis=1, how="all").copy()
    df.columns = df.columns.astype(str).str.strip()
    return df


df = load_data(INPUT_FILE, sheet_name=SHEET_NAME, header_row=HEADER_ROW)

print("Loaded dataset shape:", df.shape)
print("Columns found:", df.columns.tolist())
print("\nPreview:")
print(df.head(3))


# =========================================================
# 3. BASIC CLEANING
# =========================================================
def normalize_bool_like(series):
    s = series.astype(str).str.strip().str.lower()
    mapping = {
        "true": 1, "false": 0,
        "yes": 1, "no": 0,
        "y": 1, "n": 0,
        "1": 1, "0": 0,
        "t": 1, "f": 0,
    }
    return s.map(mapping)


bool_like_cols = [
    "feature_ai_tutor_used",
    "feature_leaderboard_viewed",
    "feature_progress_checked",
]
for col in bool_like_cols:
    if col in df.columns:
        converted = normalize_bool_like(df[col])
        if converted.notna().sum() > 0:
            df[col] = converted.fillna(0).astype(int)

if "user_id" in df.columns:
    df = df.drop_duplicates(subset=["user_id"]).copy()
else:
    df = df.drop_duplicates().copy()

for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].astype(str).str.strip()

missing_tokens = {"", "nan", "none", "null", "na", "n/a"}
for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].apply(
            lambda x: np.nan if str(x).strip().lower() in missing_tokens else x
        )

for col in df.columns:
    if df[col].dtype == "object":
        converted = pd.to_numeric(df[col], errors="coerce")
        if converted.notna().sum() >= max(3, int(0.7 * len(df))):
            df[col] = converted

for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        df[col] = df[col].fillna(df[col].median())
    else:
        mode_vals = df[col].mode(dropna=True)
        df[col] = df[col].fillna(mode_vals.iloc[0] if len(mode_vals) > 0 else "unknown")

print("\nData types after cleaning:")
print(df.dtypes)


# =========================================================
# 4. FEATURE ENGINEERING
# =========================================================
if {"sessions_last_7d", "exercises_completed_7d",
        "streak_current", "notif_open_rate_30d"}.issubset(df.columns):
    df["activeness_score"] = (
        (df["sessions_last_7d"].clip(0, 20) / 20) * 0.35 +
        (df["exercises_completed_7d"].clip(0, 20) / 20) * 0.30 +
        (df["streak_current"].clip(0, 31) / 31) * 0.20 +
        (df["notif_open_rate_30d"].clip(0, 1)) * 0.15
    ).round(3)

if {"streak_current", "coins_balance", "motivation_score"}.issubset(df.columns):
    lb_val = df["feature_leaderboard_viewed"].astype(int) \
             if "feature_leaderboard_viewed" in df.columns else 0
    df["gamification_score"] = (
        (df["streak_current"].clip(0, 31) / 31) * 0.35 +
        (df["coins_balance"].clip(0, 600) / 600) * 0.30 +
        (df["motivation_score"].clip(0, 1)) * 0.20 +
        lb_val * 0.15
    ).round(3)

if "notif_open_rate_30d" in df.columns:
    ai_val   = df["feature_ai_tutor_used"].astype(int)   if "feature_ai_tutor_used"   in df.columns else 0
    lb_val   = df["feature_leaderboard_viewed"].astype(int) if "feature_leaderboard_viewed" in df.columns else 0
    prog_val = df["feature_progress_checked"].astype(int) if "feature_progress_checked" in df.columns else 0
    df["engagement_depth"] = (
        ai_val   * 0.35 +
        lb_val   * 0.25 +
        prog_val * 0.20 +
        df["notif_open_rate_30d"].clip(0, 1) * 0.20
    ).round(3)


# =========================================================
# 5. CHURN RISK VIA LOGISTIC REGRESSION
# =========================================================
# ── 5a. Build churn target from lifecycle_stage ───────────────────────────
# Treat churned + inactive as "at risk" (binary signal for training).
# We then use the predicted probability to assign a 4-level risk label.
# The probability thresholds are derived from the data's own distribution
# (percentile-based), so no numeric cutoffs are hardcoded here.

print("\n" + "=" * 55)
print("CHURN RISK ESTIMATION — LOGISTIC REGRESSION")
print("=" * 55)

CHURN_STAGES   = {"churned", "inactive"}
ENGAGED_STAGES = {"paid", "trial"}

if "lifecycle_stage" in df.columns:
    df["_churn_target"] = df["lifecycle_stage"].str.lower().map(
        lambda s: 1 if s in CHURN_STAGES else (0 if s in ENGAGED_STAGES else np.nan)
    )
    unknown_count = df["_churn_target"].isna().sum()
    if unknown_count > 0:
        print(f"  ⚠️  {unknown_count} rows have unrecognised lifecycle_stage values "
              f"and will be excluded from churn model training.")
    df_train = df.dropna(subset=["_churn_target"]).copy()
else:
    raise ValueError(
        "Column 'lifecycle_stage' is required for churn risk estimation.\n"
        "Ensure your dataset contains this column."
    )

print(f"\n  Training set size: {len(df_train)} rows")
print("  Churn target distribution:")
print(df_train["_churn_target"].value_counts().rename({0: "engaged (0)", 1: "at-risk (1)"}))

# ── 5b. Select behavioural features for the churn model ───────────────────
# Only numeric behavioural signals — no lifecycle_stage (that would leak the target)
churn_feature_candidates = [
    "days_since_signup",
    "sessions_last_7d",
    "exercises_completed_7d",
    "streak_current",
    "coins_balance",
    "notif_open_rate_30d",
    "motivation_score",
    "activeness_score",
    "gamification_score",
    "engagement_depth",
    "feature_ai_tutor_used",
    "feature_leaderboard_viewed",
    "feature_progress_checked",
]
churn_features = [c for c in churn_feature_candidates if c in df_train.columns]

print(f"\n  Churn model features ({len(churn_features)}): {churn_features}")

X_churn = df_train[churn_features].values
y_churn = df_train["_churn_target"].astype(int).values

# ── 5c. Train logistic regression with cross-validated probability output ─
churn_scaler = StandardScaler()
X_churn_scaled = churn_scaler.fit_transform(X_churn)

lr_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",   # handles imbalanced churn/engaged ratio
    random_state=RANDOM_STATE,
    solver="lbfgs",
)

# 5-fold stratified CV → out-of-fold churn probabilities (more honest than train-set probs)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
oof_probs = cross_val_predict(
    lr_model, X_churn_scaled, y_churn,
    cv=cv, method="predict_proba"
)[:, 1]   # probability of being at-risk

# Evaluate model quality
oof_preds = (oof_probs >= 0.5).astype(int)
print("\n  Cross-validated classification report:")
print(classification_report(y_churn, oof_preds, target_names=["engaged", "at-risk"]))
print(f"  CV ROC-AUC: {roc_auc_score(y_churn, oof_probs):.4f}")

# ── 5d. Fit final model on full training set for inference on all rows ─────
lr_model.fit(X_churn_scaled, y_churn)

# Score every row (including any with unrecognised lifecycle_stage)
X_all_scaled = churn_scaler.transform(df[churn_features].values)
df["_churn_prob"] = lr_model.predict_proba(X_all_scaled)[:, 1]

# ── 5e. Assign risk labels using DATA-DRIVEN percentile cutoffs ───────────
# Percentile boundaries are computed from the actual probability distribution —
# no numeric thresholds are hardcoded here. The distribution itself decides
# where each label band begins and ends.
# With 4 labels (low / medium / high / critical) we split into 4 equal-mass bands.
n_labels  = len(CHURN_RISK_LABELS)
quantiles = np.linspace(0, 100, n_labels + 1)          # e.g. [0, 25, 50, 75, 100]
boundaries = np.percentile(df["_churn_prob"], quantiles)

print(f"\n  Churn probability percentile boundaries (from data):")
for i, (lo, hi) in enumerate(zip(boundaries[:-1], boundaries[1:])):
    label = CHURN_RISK_LABELS[i]
    print(f"    {label:<10}  p_churn ∈ [{lo:.4f}, {hi:.4f})")

def assign_churn_label(prob: float, boundaries: np.ndarray,
                        labels: list) -> str:
    """
    Maps a churn probability to a label using the data-derived boundaries.
    The last bucket is inclusive on both ends (catches the maximum value).
    """
    for i, (lo, hi) in enumerate(zip(boundaries[:-1], boundaries[1:])):
        if prob <= hi or i == len(labels) - 1:
            return labels[i]
    return labels[-1]

df["churn_risk"] = df["_churn_prob"].apply(
    lambda p: assign_churn_label(p, boundaries, CHURN_RISK_LABELS)
)

print("\n  churn_risk label distribution:")
print(df["churn_risk"].value_counts().sort_index())

# ── 5f. Store readable probability column, drop internals ─────────────────
df["churn_prob_score"] = df["_churn_prob"].round(4)
df.drop(columns=["_churn_target", "_churn_prob"], inplace=True, errors="ignore")


# =========================================================
# 6. SELECT FEATURES FOR CLUSTERING
# =========================================================
identifier_cols = ["user_id"]

behavior_numeric_candidates = [
    "days_since_signup",
    "sessions_last_7d",
    "exercises_completed_7d",
    "streak_current",
    "coins_balance",
    "preferred_hour",
    "notif_open_rate_30d",
    "motivation_score",
    "activeness_score",
    "gamification_score",
    "engagement_depth",
    "feature_ai_tutor_used",
    "feature_leaderboard_viewed",
    "feature_progress_checked",
]

profile_categorical_candidates = [
    "age_band",
    "region",
    "lifecycle_stage",
]

selected_cols = []
for col in behavior_numeric_candidates:
    if col in df.columns:
        selected_cols.append(col)

if INCLUDE_PROFILE_COLUMNS:
    for col in profile_categorical_candidates:
        if col in df.columns:
            selected_cols.append(col)

if not INCLUDE_LIFECYCLE_STAGE and "lifecycle_stage" in selected_cols:
    selected_cols.remove("lifecycle_stage")

if len(selected_cols) < 2:
    selected_cols = [c for c in df.columns if c not in identifier_cols
                     and c not in ("churn_risk", "churn_prob_score", "cluster_id")]

numeric_cols      = [c for c in selected_cols if pd.api.types.is_numeric_dtype(df[c])]
categorical_cols  = [c for c in selected_cols if not pd.api.types.is_numeric_dtype(df[c])]

print("\n\nSelected clustering feature columns:")
print(selected_cols)

if len(selected_cols) == 0:
    raise ValueError("No usable feature columns found for clustering.")


# =========================================================
# 7. PREPROCESS
# =========================================================
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(),                        numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"),  categorical_cols),
    ],
    remainder="drop",
)

X = preprocessor.fit_transform(df[selected_cols])
print("\nTransformed feature matrix shape:", X.shape)


# =========================================================
# 8. FIND BEST K (SILHOUETTE)
# =========================================================
n_samples = len(df)
k_min, k_max = 2, min(10, n_samples - 1)

best_k, best_score = None, -1
print("\n=== Searching best K via silhouette score ===")

for k in range(k_min, k_max + 1):
    km     = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20)
    labels = km.fit_predict(X)
    score  = silhouette_score(X, labels) if len(set(labels)) > 1 else -1
    print(f"  K = {k}  silhouette = {score:.4f}")
    if score > best_score:
        best_score, best_k = score, k

print(f"\nBest K: {best_k}  (silhouette = {best_score:.4f})")


# =========================================================
# 9. FINAL K-MEANS
# =========================================================
final_km = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=30)
df["cluster_id"] = final_km.fit_predict(X)


# =========================================================
# 10. CLUSTER SUMMARY
# =========================================================
summary_frames = []

if len(numeric_cols) > 0:
    summary_frames.append(df.groupby("cluster_id")[numeric_cols].mean().round(3))

if len(categorical_cols) > 0:
    cat_summary = pd.DataFrame(index=sorted(df["cluster_id"].unique()))
    for col in categorical_cols:
        cat_summary[f"dominant_{col}"] = (
            df.groupby("cluster_id")[col]
              .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else "unknown")
        )
    summary_frames.append(cat_summary)

# Include churn_risk distribution per cluster
churn_dist = (
    df.groupby(["cluster_id", "churn_risk"])
      .size()
      .unstack(fill_value=0)
      .add_prefix("churn_")
)
summary_frames.append(churn_dist)
summary_frames.append(df.groupby("cluster_id").size().rename("user_count"))

cluster_summary = pd.concat(summary_frames, axis=1)

print("\n=== Cluster Summary ===")
print(cluster_summary)


# =========================================================
# 11. SEGMENT NAMES (BEHAVIOUR-DRIVEN)
# =========================================================
def assign_segment_name(row):
    if "activeness_score" in row.index and pd.notna(row["activeness_score"]):
        val = row["activeness_score"]
        if val >= 0.70:
            return "Highly Active Users"
        elif val >= 0.40:
            return "Moderately Active Users"
        else:
            return "Low Activity Users"

    if "sessions_last_7d" in row.index and pd.notna(row["sessions_last_7d"]):
        val = row["sessions_last_7d"]
        if val >= 8:
            return "Frequent Users"
        elif val >= 3:
            return "Regular Users"
        else:
            return "Low Engagement Users"

    return "Behavioral Segment"


segment_name_map = {
    cid: assign_segment_name(cluster_summary.loc[cid])
    for cid in cluster_summary.index
}
df["segment_name"] = df["cluster_id"].map(segment_name_map)


# =========================================================
# 12. SAVE OUTPUT
# =========================================================
df.to_csv("user_segments.csv", index=False)

print("\nSaved: user_segments.csv")

# ── Show churn risk breakdown per segment ─────────────────────────────────
print("\n=== Churn Risk Distribution per Segment ===")
print(
    df.groupby(["segment_name", "churn_risk"])
      .size()
      .unstack(fill_value=0)
      .to_string()
)

# ── Sample output ──────────────────────────────────────────────────────────
show_cols  = [c for c in ["user_id", "cluster_id", "segment_name",
                           "churn_risk", "churn_prob_score"] if c in df.columns]
show_cols += [c for c in selected_cols if c not in show_cols][:5]

print("\n=== Sample Segmented Users ===")
print(df[show_cols].head(15).to_string(index=False))

Loaded dataset shape: (2000, 15)
Columns found: ['user_id', 'lifecycle_stage', 'days_since_signup', 'age_band', 'region', 'sessions_last_7d', 'exercises_completed_7d', 'streak_current', 'coins_balance', 'feature_ai_tutor_used', 'feature_leaderboard_viewed', 'feature_progress_checked', 'preferred_hour', 'notif_open_rate_30d', 'motivation_score']

Preview:
     user_id lifecycle_stage  days_since_signup age_band region  \
0  SYN_00001        inactive                 76    18-24  tier1   
1  SYN_00002           trial                  3    18-24  tier2   
2  SYN_00003         churned                104    18-24  tier1   

   sessions_last_7d  exercises_completed_7d  streak_current  coins_balance  \
0                 0                       0               0          20.19   
1                10                       9               0          90.72   
2                 0                       0               0          27.15   

   feature_ai_tutor_used  feature_leaderboard_viewed  \
0    

In [ ]:
import os
os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')

In [ ]:
import json
import re
from pathlib import Path
from typing import Dict, List, Optional


OUTPUT_DIR = "outputs"


def find_input_file() -> Path:
    """
    Automatically find the SpeakX knowledge bank markdown file.
    """
    exact_names = [
        "speakx_knowledge_bank.md",
        "speakx knowledge bank.md",
        "knowledge_bank.md",
    ]

    search_roots = [
        Path("."),
        Path("/mnt/data"),
        Path("/content"),
    ]

    # 1. Try exact filenames first
    for root in search_roots:
        if root.exists():
            for name in exact_names:
                candidate = root / name
                if candidate.exists():
                    return candidate

    # 2. Try recursive search for close matches
    patterns = [
        "*speakx*knowledge*bank*.md",
        "*knowledge*bank*.md",
        "*.md",
    ]

    for root in search_roots:
        if root.exists():
            for pattern in patterns:
                matches = list(root.rglob(pattern))
                if matches:
                    # Prefer files with "speakx" in name
                    matches.sort(key=lambda p: ("speakx" not in p.name.lower(), len(str(p))))
                    return matches[0]

    raise FileNotFoundError(
        "Could not find the input markdown file.\n"
        "Put your file in the current folder or /mnt/data, and name it like:\n"
        "speakx_knowledge_bank.md"
    )


def read_text(path: Path) -> str:
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")
    return path.read_text(encoding="utf-8")


def clean(text: Optional[str]) -> Optional[str]:
    if text is None:
        return None
    text = re.sub(r"\s+", " ", text).strip()
    return text or None


def extract_section(text: str, heading: str) -> str:
    pattern = rf"^## {re.escape(heading)}\s*$([\s\S]*?)(?=^## |\Z)"
    match = re.search(pattern, text, flags=re.MULTILINE)
    return match.group(1).strip() if match else ""


def extract_subsection(text: str, heading: str) -> str:
    pattern = rf"^### {re.escape(heading)}\s*$([\s\S]*?)(?=^### |\Z)"
    match = re.search(pattern, text, flags=re.MULTILINE)
    return match.group(1).strip() if match else ""


def parse_bullet_list(block: str) -> List[str]:
    items = []
    for line in block.splitlines():
        line = line.strip()
        if line.startswith("- "):
            items.append(clean(line[2:]))
    return [x for x in items if x]


def parse_north_star(md: str) -> Dict:
    section = extract_section(md, "North Star Metric")

    primary_metric = None
    m = re.search(r"\*\*Primary Metric\*\*:\s*(.+)", section)
    if m:
        primary_metric = clean(m.group(1))

    justification = []
    before_supporting = section.split("**Supporting Metrics:**")[0]
    for line in before_supporting.splitlines():
        line = line.strip()
        if line.startswith("- "):
            justification.append(clean(line[2:]))

    supporting_metrics = []
    after_supporting = ""
    if "**Supporting Metrics:**" in section:
        after_supporting = section.split("**Supporting Metrics:**", 1)[1]

    definition_map = {
        "Trial → Paid conversion rate": "Percentage of trial users who become paying subscribers.",
        "W1 Retention post-conversion": "Percentage of converted users retained through the first week after conversion.",
        "Monthly Retention (D30+)": "Percentage of users retained after 30+ days.",
        "Average practice sessions per active week": "Average number of practice sessions completed by an active learner in a week.",
    }

    reason_map = {
        "Trial → Paid conversion rate": "Measures monetization efficiency from the trial experience.",
        "W1 Retention post-conversion": "Shows whether the early paid experience creates stickiness.",
        "Monthly Retention (D30+)": "Tracks sustained product value over time.",
        "Average practice sessions per active week": "Measures consistency and habit strength.",
    }

    for line in after_supporting.splitlines():
        line = line.strip()
        if not line.startswith("- "):
            continue

        item = line[2:].strip()
        metric_match = re.match(r"(.+?)\s*\(Target:\s*(.+?)\)", item)

        if metric_match:
            metric_name = clean(metric_match.group(1))
            target = clean(metric_match.group(2))
        else:
            metric_name = clean(item)
            target = None

        supporting_metrics.append({
            "metric": metric_name,
            "definition": definition_map.get(metric_name),
            "target": target,
            "reason": reason_map.get(metric_name),
        })

    measurable_proxy = {
        "name": primary_metric,
        "definition": "Count of monthly active learners who maintain a 7+ day practice streak.",
        "formula": "Monthly active learners with streak >= 7 days",
        "time_window": "Monthly",
        "reason": "The knowledge bank explicitly defines this as the primary metric for sustained engagement and habit formation.",
    }

    return {
        "north_star_metric": primary_metric,
        "justification": justification,
        "measurable_proxy": measurable_proxy,
        "supporting_metrics": supporting_metrics,
        "confidence_note": "High confidence because the knowledge bank explicitly names the primary metric and supporting metrics.",
    }


def infer_lifecycle_stage(feature: Optional[str], expected_outcome: Optional[str], success_metric: Optional[str]) -> Optional[str]:
    text = " ".join([feature or "", expected_outcome or "", success_metric or ""]).lower()

    if any(word in text for word in ["trial", "paid", "conversion", "subscribe"]):
        return "conversion"
    if any(word in text for word in ["retention", "streak", "weekly active", "monthly retention", "active week"]):
        return "retention"
    if any(word in text for word in ["adoption", "feature use", "session completion", "practice session"]):
        return "activation"
    if any(word in text for word in ["re-engage", "come back", "return"]):
        return "re-engagement"

    return "activation"


def parse_feature_goal_map(md: str) -> Dict:
    section = extract_section(md, "Feature → Goal → Outcome Mapping")
    lines = [line.rstrip() for line in section.splitlines() if line.strip()]

    rows = []
    in_table = False

    for line in lines:
        if line.startswith("| Feature |"):
            in_table = True
            continue
        if in_table and line.startswith("|---"):
            continue
        if in_table and line.startswith("|"):
            parts = [p.strip() for p in line.strip("|").split("|")]
            if len(parts) >= 4:
                feature, user_goal, expected_outcome, success_metric = parts[:4]
                feature = clean(feature)
                user_goal = clean(user_goal)
                expected_outcome = clean(expected_outcome)
                success_metric = clean(success_metric)

                rows.append({
                    "feature": feature,
                    "lifecycle_stage": infer_lifecycle_stage(feature, expected_outcome, success_metric),
                    "user_goal": user_goal,
                    "expected_outcome": expected_outcome,
                    "success_metric": success_metric,
                    "evidence": [
                        f"{feature} → {user_goal} → {expected_outcome} → {success_metric}"
                    ],
                })

    return {"feature_goal_outcome_map": rows}


def parse_allowed_tones(md: str) -> List[Dict]:
    section = extract_section(md, "Ethical Communication Guidelines")
    allowed_block = extract_subsection(section, "Allowed Tones & Themes")

    chunks = re.split(r"(?=\*\*.+?\*\*\s*✓)", allowed_block)
    tones = []

    desc_map = {
        "Empowerment": "Affirms learner growth, agency, and confidence-building.",
        "Accomplishment": "Highlights milestones, progress, and achievements.",
        "Encouragement": "Supports continued practice with positive reinforcement.",
        "Social Influence": "Uses peers or broader community as positive motivation.",
        "Scarcity (Positive)": "Creates urgency without fear or manipulation.",
        "Ownership": "Frames progress, balance, or pathway as belonging to the learner.",
    }

    allowed_for_map = {
        "Empowerment": ["confidence building", "habit reinforcement", "progress updates"],
        "Accomplishment": ["milestone messages", "streak celebrations", "progress notifications"],
        "Encouragement": ["daily reminders", "re-engagement", "consistency nudges"],
        "Social Influence": ["leaderboards", "community features", "peer-based motivation"],
        "Scarcity (Positive)": ["trial conversion", "limited-time opportunities", "live sessions"],
        "Ownership": ["coins", "progress reports", "learning path reminders"],
    }

    disallowed_for_map = {
        "Empowerment": ["shame-based reminders"],
        "Accomplishment": ["false achievement claims"],
        "Encouragement": ["pressure tactics"],
        "Social Influence": ["harmful comparison", "humiliation"],
        "Scarcity (Positive)": ["fear-heavy loss framing", "deceptive urgency"],
        "Ownership": ["guilt-heavy possession framing"],
    }

    for chunk in chunks:
        chunk = chunk.strip()
        if not chunk.startswith("**"):
            continue

        header_match = re.match(r"\*\*(.+?)\*\*\s*✓", chunk)
        if not header_match:
            continue

        tone = clean(header_match.group(1))
        bullets = parse_bullet_list(chunk)

        tones.append({
            "tone": tone,
            "description": desc_map.get(tone),
            "allowed_for": allowed_for_map.get(tone, []),
            "disallowed_for": disallowed_for_map.get(tone, []),
            "evidence": bullets[:3],
        })

    return tones


def parse_disallowed_tones(md: str) -> List[Dict]:
    section = extract_section(md, "Ethical Communication Guidelines")
    disallowed_block = extract_subsection(section, "Disallowed Tones & Themes")

    if not disallowed_block:
        return []

    chunks = re.split(r"(?=\*\*.+?\*\*)", disallowed_block)
    results = []

    for chunk in chunks:
        chunk = chunk.strip()
        if not chunk.startswith("**"):
            continue

        header_match = re.match(r"\*\*(.+?)\*\*", chunk)
        if not header_match:
            continue

        tone = clean(header_match.group(1))
        bullets = parse_bullet_list(chunk)
        reason = bullets[0] if bullets else None

        results.append({
            "tone": tone,
            "reason": reason,
            "evidence": bullets[:3],
        })

    return results


def infer_stages_from_hook(title: Optional[str]) -> List[str]:
    if not title:
        return []

    lower = title.lower()

    if "accomplishment" in lower:
        return ["activation", "retention"]
    if "social influence" in lower:
        return ["activation", "retention"]
    if "scarcity" in lower:
        return ["conversion", "re-engagement"]
    if "unpredictability" in lower:
        return ["activation", "retention"]
    if "loss" in lower:
        return ["re-engagement", "retention"]
    if "ownership" in lower:
        return ["retention", "conversion"]
    if "epic meaning" in lower:
        return ["awareness", "activation"]
    if "creativity" in lower:
        return ["activation", "retention"]

    return []


def parse_hooks(md: str) -> List[Dict]:
    section = extract_section(md, "Octolysis Framework Hooks")

    pattern = r"^### (.+?)\s*$([\s\S]*?)(?=^### |\Z)"
    matches = re.finditer(pattern, section, flags=re.MULTILINE)

    hooks = []
    for match in matches:
        title = clean(match.group(1))
        body = match.group(2)

        app = re.search(r"\*\*Scenario Application\*\*:\s*(.+)", body)
        msg = re.search(r"\*\*Message Hook\*\*:\s*(.+)", body)
        goal = re.search(r"\*\*Goal Alignment\*\*:\s*(.+)", body)

        scenario_text = clean(app.group(1)) if app else None
        message_hook = clean(msg.group(1)) if msg else None
        goal_alignment = clean(goal.group(1)) if goal else None

        best_features = []
        if scenario_text:
            best_features = [x.strip() for x in scenario_text.split(",") if x.strip()]

        hooks.append({
            "hook": message_hook,
            "type": title,
            "description": goal_alignment,
            "best_for_stages": infer_stages_from_hook(title),
            "best_for_features": best_features,
            "risk_notes": "Use ethically and avoid manipulative framing.",
            "evidence": [x for x in [scenario_text, message_hook, goal_alignment] if x],
        })

    return hooks


def build_outputs(md: str) -> Dict[str, Dict]:
    return {
        "company_north_star": parse_north_star(md),
        "feature_goal_map": parse_feature_goal_map(md),
        "allowed_tone_hook_matrix": {
            "allowed_tones": parse_allowed_tones(md),
            "disallowed_tones": parse_disallowed_tones(md),
            "hook_taxonomy": parse_hooks(md),
        },
    }


def save_outputs(outputs: Dict[str, Dict], output_dir: Path) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)

    file_map = {
        "company_north_star.json": outputs["company_north_star"],
        "feature_goal_map.json": outputs["feature_goal_map"],
        "allowed_tone_hook_matrix.json": outputs["allowed_tone_hook_matrix"],
    }

    for filename, payload in file_map.items():
        path = output_dir / filename
        path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
        print(f"saved: {path}")


def main() -> None:
    input_file = find_input_file()
    output_dir = Path(OUTPUT_DIR)

    print(f"Found input file: {input_file}")
    md = read_text(input_file)
    outputs = build_outputs(md)
    save_outputs(outputs, output_dir)

    print("\nDone. Generated files:")
    for p in output_dir.glob("*.json"):
        print(p)


main()

Found input file: speakx_knowledge_bank.md
saved: outputs/company_north_star.json
saved: outputs/feature_goal_map.json
saved: outputs/allowed_tone_hook_matrix.json

Done. Generated files:
outputs/allowed_tone_hook_matrix.json
outputs/company_north_star.json
outputs/feature_goal_map.json


In [ ]:
import json

with open("company_north_star.json", "r", encoding="utf-8") as f:
    print(json.load(f))

{'north_star_metric': 'Monthly Active Learners (MAL) with 7+ Day Practice Streak', 'justification': ['Measures sustained engagement and habit formation', 'Indicates genuine learning progress versus casual usage', 'Correlates with 3x higher conversion rates and 5x better retention', 'Trial → Paid conversion rate (Target: 12-18%)', 'W1 Retention post-conversion (Target: 65-75%)', 'Monthly Retention (D30+) (Target: 45-60%)', 'Average practice sessions per active week (Target: 4.2+)'], 'measurable_proxy': {'name': 'Monthly Active Learners (MAL) with 7+ Day Practice Streak', 'definition': 'Count of monthly active learners who maintain a 7+ day practice streak.', 'formula': 'Monthly active learners with streak >= 7 days', 'time_window': 'Monthly', 'reason': 'The knowledge bank explicitly defines this as the primary metric for sustained engagement and habit formation.'}, 'supporting_metrics': [], 'confidence_note': 'High confidence because the knowledge bank explicitly names the primary metri

In [ ]:
with open("feature_goal_map.json", "r", encoding="utf-8") as f:
    print(json.load(f))

{'metadata': {'company': 'SpeakX', 'project': 'Project Aurora — Kriti 2026', 'task': 'Task 1 — System Architecture & Intelligence Design', 'deliverable': 'feature_goal_map.json', 'description': 'Feature → Lifecycle → Outcome mapping derived from 2,000-user synthetic behavioral dataset', 'dataset_summary': {'total_users': 2000, 'lifecycle_distribution': {'trial': 551, 'paid': 961, 'churned': 221, 'inactive': 267}, 'demographics': {'age_bands': {'18-24': 1061, '25-34': 482, '35-44': 457}, 'regions': {'tier1': 1065, 'tier2': 457, 'tier3': 478}}, 'preferred_hour': {'mean': 15.1, 'median': 17.0, 'implication': 'Users skew evening — peak engagement window is 15:00–21:00'}}, 'north_star_metric': 'W1_retention_rate', 'north_star_proxy': 'exercises_completed in first 7 days post trial-to-monthly conversion', 'north_star_justification': 'Paid users who exercise in W1 show avg streak of 14.06 and coins of 325 vs near-zero for churned/inactive, confirming early exercise habit as the strongest pred

In [ ]:
import argparse
import pandas as pd
import google.generativeai as genai


GOAL_SCHEMA = {
    "segment_id": "<string>",
    "segment_name": "<string>",
    "lifecycle_stage": "<trial|paid|churned|inactive>",
    "primary_goal": "<one sentence, max 15 words, must reference north star>",
    "sub_goals": [
        "<sub-goal 1, max 20 words>",
        "<sub-goal 2, max 20 words>",
        "<sub-goal 3, max 20 words>"
    ],
    "day_on_day_focus": {
        "DAY_KEY": "<focus activity or message intent for that day, max 15 words>"
    },
    "recommended_features": [
        "<feature name from feature_goal_map>"
    ],
    "north_star_lever": "<which proxy metric this primarily moves>",
    "core_drive": "<primary Octolysis drive from data>",
    "success_metric": "<measurable KPI tied to feature_goal_map outcomes>",
    "communication_priority": "<high|medium|low>",
    "churn_risk_response_strategy": "<how goals adapt to churn_risk label>",
    "north_star_alignment_note": "<one sentence explaining north star alignment>",
}


def setup_gemini():
    api_key = os.environ.get("GEMINI_API_KEY")
    if not api_key:
        raise EnvironmentError(
            "GEMINI_API_KEY not found.\n"
            "Run:\n"
            "import os\n"
            "os.environ['GEMINI_API_KEY'] = 'YOUR_API_KEY'"
        )

    genai.configure(api_key=api_key)

    return genai.GenerativeModel(
        model_name="gemini-1.5-flash",
        generation_config=genai.GenerationConfig(
            temperature=0.7,
            max_output_tokens=1000,
            response_mime_type="application/json",
        ),
    )


def list_current_files():
    print("\nCurrent working directory:", os.getcwd())
    print("Available files:")
    for f in sorted(os.listdir(".")):
        print(" -", f)
    print()


def resolve_file(path: str, label: str) -> str:
    if os.path.exists(path):
        return path

    base = os.path.basename(path)
    if os.path.exists(base):
        return base

    raise FileNotFoundError(
        f"{label} file not found: {path}\n"
        f"Current working directory: {os.getcwd()}\n"
        f"Available files: {sorted(os.listdir('.'))}"
    )


def load_north_star(path: str) -> dict:
    path = resolve_file(path, "North Star")
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    print(f"✅ Loaded North Star file: {path}")
    print(f"✅ North Star metric: {data.get('north_star_metric')}")
    return data


def load_feature_map(path: str) -> list:
    path = resolve_file(path, "Feature Map")
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if isinstance(data, dict):
        entries = data.get("feature_goal_outcome_map", [])
    elif isinstance(data, list):
        entries = data
    else:
        entries = []

    print(f"✅ Loaded Feature Map file: {path}")
    print(f"✅ Total feature-goal mappings: {len(entries)}")
    return entries


def load_and_aggregate_segments(path: str):
    path = resolve_file(path, "Segments")
    df = pd.read_csv(path)
    print(f"✅ Loaded {len(df)} user rows from: {path}")

    print("\nDetected columns in CSV:")
    print(df.columns.tolist())

    # -------------------------------------------------
    # Rename columns from your CSV to expected names
    # -------------------------------------------------
    rename_map = {
        "cluster_id": "segment_id",
        "gamification_score": "gamification_propensity",
        "feature_ai_tutor_used": "ai_tutor_propensity",
        "feature_leaderboard_viewed": "leaderboard_propensity",
    }
    df = df.rename(columns=rename_map)

    # -------------------------------------------------
    # Required columns from your actual file
    # -------------------------------------------------
    if "segment_id" not in df.columns:
        raise ValueError("segment_id could not be created from cluster_id")

    if "segment_name" not in df.columns:
        df["segment_name"] = "Unknown Segment"

    if "lifecycle_stage" not in df.columns:
        df["lifecycle_stage"] = "trial"

    # Make segment_id readable like SEG_00, SEG_01 ...
    df["segment_id"] = df["segment_id"].astype(int).apply(lambda x: f"SEG_{x:02d}")

    # -------------------------------------------------
    # Create churn_risk dynamically because your CSV
    # does not contain churn_risk column
    # -------------------------------------------------
    df["lifecycle_stage"] = df["lifecycle_stage"].astype(str).str.strip().str.lower()

    if "activeness_score" not in df.columns:
        df["activeness_score"] = 0

    # Quantile-based churn risk from activeness_score
    # lower activeness => higher churn risk
    try:
        risk_bins = pd.qcut(
            df["activeness_score"].rank(method="first"),
            q=4,
            labels=["critical", "high", "medium", "low"]
        )
        df["churn_risk"] = risk_bins.astype(str)
    except Exception:
        df["churn_risk"] = "medium"

    # Override risk using lifecycle stage where strongly implied
    df.loc[df["lifecycle_stage"] == "churned", "churn_risk"] = "critical"
    df.loc[df["lifecycle_stage"] == "inactive", "churn_risk"] = "high"

    # -------------------------------------------------
    # Create missing columns expected later in pipeline
    # -------------------------------------------------
    if "segment_description" not in df.columns:
        df["segment_description"] = df["segment_name"]

    if "user_id" not in df.columns:
        df["user_id"] = [f"USER_{i+1:05d}" for i in range(len(df))]

    if "sessions_last_7d" not in df.columns:
        df["sessions_last_7d"] = 0

    if "exercises_completed_7d" not in df.columns:
        df["exercises_completed_7d"] = 0

    if "streak_current" not in df.columns:
        df["streak_current"] = 0

    if "motivation_score" not in df.columns:
        df["motivation_score"] = 0

    if "notif_open_rate_30d" not in df.columns:
        df["notif_open_rate_30d"] = 0

    if "gamification_propensity" not in df.columns:
        df["gamification_propensity"] = 0

    if "ai_tutor_propensity" not in df.columns:
        df["ai_tutor_propensity"] = 0

    if "leaderboard_propensity" not in df.columns:
        df["leaderboard_propensity"] = 0

    # Your CSV has no direct social metric
    if "social_propensity" not in df.columns:
        df["social_propensity"] = 0

    # Derive Octolysis-like drive from strongest behavior signal
    def infer_drive(row):
        scores = {
            "Accomplishment": row.get("leaderboard_propensity", 0),
            "Empowerment": row.get("ai_tutor_propensity", 0),
            "Ownership": row.get("gamification_propensity", 0),
        }
        return max(scores, key=scores.get)

    if "primary_octolysis_drive" not in df.columns:
        df["primary_octolysis_drive"] = df.apply(infer_drive, axis=1)

    # Derive a simple notif frequency from lifecycle / activeness
    def infer_notif_freq(row):
        stage = str(row.get("lifecycle_stage", "")).lower()
        act = float(row.get("activeness_score", 0) or 0)

        if stage == "churned":
            return 1
        if stage == "inactive":
            return 1
        if stage == "trial":
            return 2
        if act >= df["activeness_score"].median():
            return 2
        return 1

    if "recommended_notif_freq_per_day" not in df.columns:
        df["recommended_notif_freq_per_day"] = df.apply(infer_notif_freq, axis=1)

    # -------------------------------------------------
    # Final cleanup
    # -------------------------------------------------
    df["churn_risk"] = df["churn_risk"].astype(str).str.strip().str.lower()
    df["lifecycle_stage"] = df["lifecycle_stage"].astype(str).str.strip().str.lower()

    churn_risk_labels = sorted(df["churn_risk"].dropna().unique().tolist())
    print(f"✅ Churn risk labels created: {churn_risk_labels}")

    def safe_mode(series):
        series = series.dropna()
        if len(series) == 0:
            return None
        mode_vals = series.mode()
        return mode_vals.iloc[0] if len(mode_vals) > 0 else None

    agg = (
        df.groupby(
            ["segment_id", "segment_name", "lifecycle_stage", "churn_risk", "segment_description"],
            dropna=False
        )
        .agg(
            user_count=("user_id", "count"),
            avg_activeness=("activeness_score", "mean"),
            avg_gamification=("gamification_propensity", "mean"),
            avg_ai_tutor=("ai_tutor_propensity", "mean"),
            avg_leaderboard=("leaderboard_propensity", "mean"),
            avg_social=("social_propensity", "mean"),
            avg_sessions_7d=("sessions_last_7d", "mean"),
            avg_exercises_7d=("exercises_completed_7d", "mean"),
            avg_streak=("streak_current", "mean"),
            avg_motivation=("motivation_score", "mean"),
            avg_notif_open_rate=("notif_open_rate_30d", "mean"),
            top_octolysis_drive=("primary_octolysis_drive", safe_mode),
            notif_freq=("recommended_notif_freq_per_day", safe_mode),
        )
        .reset_index()
    )

    print(f"✅ Aggregated combinations: {len(agg)}")
    return agg, churn_risk_labels



def get_day_keys(lifecycle_stage: str):
    if lifecycle_stage == "trial":
        return ["D0", "D1", "D2", "D3", "D4", "D5", "D6", "D7"]
    elif lifecycle_stage == "paid":
        return ["D8", "D10", "D12", "D15", "D18", "D20", "D25", "D30"]
    else:
        return ["D31", "D35", "D40", "D45", "D50", "D55", "D60", "D90"]


def get_relevant_features(feature_map: list, lifecycle_stage: str):
    stage_map = {
        "trial": ["activation"],
        "paid": ["activation", "conversion", "retention"],
        "churned": ["retention"],
        "inactive": ["retention"],
    }

    allowed = stage_map.get(lifecycle_stage, ["activation", "retention"])
    filtered = []

    for item in feature_map:
        if isinstance(item, dict) and item.get("lifecycle_stage") in allowed:
            filtered.append(item)

    return filtered if filtered else feature_map


def build_prompt(row: dict, north_star: dict, feature_map: list, churn_risk_labels: list) -> str:
    lifecycle = str(row["lifecycle_stage"]).strip().lower()
    day_keys = get_day_keys(lifecycle)
    relevant_features = get_relevant_features(feature_map, lifecycle)

    schema = dict(GOAL_SCHEMA)
    schema["day_on_day_focus"] = {d: "<focus, max 15 words>" for d in day_keys}

    churn_label = str(row.get("churn_risk", "unknown")).strip().lower()

    north_star_summary = {
        "metric": north_star.get("north_star_metric"),
        "proxy_definition": north_star.get("measurable_proxy", {}).get("definition"),
        "proxy_formula": north_star.get("measurable_proxy", {}).get("formula"),
        "justification": north_star.get("justification", []),
    }

    feature_summary = []
    for f in relevant_features:
        if isinstance(f, dict):
            feature_summary.append({
                "feature": f.get("feature"),
                "user_goal": f.get("user_goal"),
                "expected_outcome": f.get("expected_outcome"),
                "success_metric": f.get("success_metric"),
            })

    profile = {
        "segment_id": row.get("segment_id"),
        "segment_name": row.get("segment_name"),
        "segment_description": row.get("segment_description", ""),
        "lifecycle_stage": lifecycle,
        "user_count_in_cohort": int(row.get("user_count", 0)),
        "churn_risk": churn_label,
        "avg_activeness_score": round(float(row.get("avg_activeness", 0) or 0), 3),
        "avg_gamification_propensity": round(float(row.get("avg_gamification", 0) or 0), 3),
        "avg_ai_tutor_propensity": round(float(row.get("avg_ai_tutor", 0) or 0), 3),
        "avg_leaderboard_propensity": round(float(row.get("avg_leaderboard", 0) or 0), 3),
        "avg_social_propensity": round(float(row.get("avg_social", 0) or 0), 3),
        "avg_sessions_last_7d": round(float(row.get("avg_sessions_7d", 0) or 0), 2),
        "avg_exercises_completed_7d": round(float(row.get("avg_exercises_7d", 0) or 0), 2),
        "avg_current_streak": round(float(row.get("avg_streak", 0) or 0), 2),
        "avg_motivation_score": round(float(row.get("avg_motivation", 0) or 0), 3),
        "avg_notif_open_rate_30d": round(float(row.get("avg_notif_open_rate", 0) or 0), 3),
        "top_octolysis_drive": row.get("top_octolysis_drive", ""),
        "recommended_notif_freq": row.get("notif_freq", ""),
    }

    prompt = f"""
You are an expert growth and engagement strategist for SpeakX, an AI-powered English speaking app.

Generate exactly one JSON object for the given segment × lifecycle combination.

Rules:
- Use churn_risk strictly as a label, not as a numeric threshold.
- Anchor the goal to the North Star metric.
- Recommended features must come from the feature list below.
- Prefer the segment's top Octolysis drive from the data.
- Return only valid JSON matching the schema.

NORTH STAR:
{json.dumps(north_star_summary, indent=2)}

FEATURE MAP:
{json.dumps(feature_summary, indent=2)}

SEGMENT PROFILE:
{json.dumps(profile, indent=2)}

REQUIRED OUTPUT SCHEMA:
{json.dumps(schema, indent=2)}

VALID CHURN LABELS IN THIS DATASET:
{json.dumps(churn_risk_labels)}

THIS SEGMENT CHURN LABEL:
{json.dumps(churn_label)}

CONSTRAINTS:
- day_on_day_focus keys must be exactly: {json.dumps(day_keys)}
- primary_goal max 15 words
- each sub_goal max 20 words
- each day_on_day_focus value max 15 words
- churn_risk_response_strategy must mention label "{churn_label}"
- output only one JSON object
""".strip()

    return prompt


def clean_json_text(raw: str) -> str:
    raw = raw.strip()
    if raw.startswith("```"):
        parts = raw.split("```")
        if len(parts) > 1:
            raw = parts[1]
        if raw.lower().startswith("json"):
            raw = raw[4:]
    return raw.strip()


def call_gemini(model, prompt: str, label: str):
    try:
        response = model.generate_content(prompt)
        raw = getattr(response, "text", "")
        raw = clean_json_text(raw)
        return json.loads(raw)
    except Exception as e:
        print(f"\n⚠️ Gemini error for [{label}]: {e}")
        return None


def generate_goal(model, row: dict, north_star: dict, feature_map: list, churn_risk_labels: list):
    label = f"{row.get('segment_id')} x {row.get('lifecycle_stage')}"
    prompt = build_prompt(row, north_star, feature_map, churn_risk_labels)
    result = call_gemini(model, prompt, label)

    if result is None:
        result = {
            "segment_id": row.get("segment_id"),
            "segment_name": row.get("segment_name"),
            "lifecycle_stage": row.get("lifecycle_stage"),
            "primary_goal": "ERROR: Gemini response could not be parsed",
            "sub_goals": [],
            "day_on_day_focus": {},
            "recommended_features": [],
            "north_star_lever": "unknown",
            "core_drive": row.get("top_octolysis_drive", "unknown"),
            "success_metric": "unknown",
            "communication_priority": "low",
            "churn_risk_response_strategy": f"Fallback used for churn label {row.get('churn_risk', 'unknown')}",
            "north_star_alignment_note": "unknown",
        }

    return result


def flatten_goal(goal: dict, source_row: dict):
    sub_goals = goal.get("sub_goals") or []
    features = goal.get("recommended_features") or []

    base = {
        "segment_id": goal.get("segment_id"),
        "segment_name": goal.get("segment_name"),
        "lifecycle_stage": goal.get("lifecycle_stage"),
        "user_count_in_cohort": source_row.get("user_count"),
        "churn_risk": source_row.get("churn_risk"),
        "avg_activeness_score": round(float(source_row.get("avg_activeness", 0) or 0), 3),
        "top_octolysis_drive": source_row.get("top_octolysis_drive"),
        "recommended_notif_freq": source_row.get("notif_freq"),
        "primary_goal": goal.get("primary_goal"),
        "sub_goal_1": sub_goals[0] if len(sub_goals) > 0 else "",
        "sub_goal_2": sub_goals[1] if len(sub_goals) > 1 else "",
        "sub_goal_3": sub_goals[2] if len(sub_goals) > 2 else "",
        "recommended_features": " | ".join(features),
        "north_star_lever": goal.get("north_star_lever"),
        "core_drive": goal.get("core_drive"),
        "success_metric": goal.get("success_metric"),
        "communication_priority": goal.get("communication_priority"),
        "churn_risk_response_strategy": goal.get("churn_risk_response_strategy"),
        "north_star_alignment_note": goal.get("north_star_alignment_note"),
    }

    day_map = goal.get("day_on_day_focus") or {}
    if day_map:
        rows = []
        for day, focus in day_map.items():
            rows.append({**base, "day": day, "day_focus": focus})
        return rows

    return [{**base, "day": "", "day_focus": ""}]


def run(
    segments_path="user_segments.csv",
    north_star_path="company_north_star.json",
    feature_map_path="feature_goal_map.json",
    output_path="segment_goals.csv",
    lifecycle_filter=None,
):
    print("\n🚀 Segment Goal Generator — Project Aurora")
    print("=" * 60)

    list_current_files()

    model = setup_gemini()
    north_star = load_north_star(north_star_path)
    feature_map = load_feature_map(feature_map_path)
    segments_df, churn_risk_labels = load_and_aggregate_segments(segments_path)

    if lifecycle_filter:
        lifecycle_filter = [x.strip().lower() for x in lifecycle_filter]
        segments_df = segments_df[segments_df["lifecycle_stage"].isin(lifecycle_filter)]
        print(f"✅ Filtered rows for lifecycle stages: {lifecycle_filter}")
        print(f"✅ Remaining combinations: {len(segments_df)}")

    total = len(segments_df)
    if total == 0:
        raise ValueError("No segment × lifecycle combinations found after filtering.")

    print(f"\n🔁 Generating goals for {total} combinations...\n")

    all_rows = []

    for i, (_, row) in enumerate(segments_df.iterrows(), start=1):
        label = f"{row['segment_id']} ({row['segment_name']}) x {row['lifecycle_stage']}"
        print(f"[{i:>2}/{total}] {label} ... ", end="")

        goal = generate_goal(
            model=model,
            row=row.to_dict(),
            north_star=north_star,
            feature_map=feature_map,
            churn_risk_labels=churn_risk_labels,
        )

        all_rows.extend(flatten_goal(goal, row.to_dict()))
        print("✓")

    out_df = pd.DataFrame(all_rows)
    out_df.to_csv(output_path, index=False)

    print(f"\n✅ Saved output to: {output_path}")
    print(f"✅ Output rows: {len(out_df)}")
    print(f"✅ Output columns: {list(out_df.columns)}")

    preview_cols = [
        "segment_id",
        "segment_name",
        "lifecycle_stage",
        "churn_risk",
        "day",
        "primary_goal",
        "day_focus",
    ]
    preview_cols = [c for c in preview_cols if c in out_df.columns]

    print("\nPreview:")
    print(out_df[preview_cols].head(8).to_string(index=False))

    return out_df


def main():
    parser = argparse.ArgumentParser(description="Generate segment goals CSV using Gemini.")

    parser.add_argument("--segments", default="user_segments.csv")
    parser.add_argument("--north_star", default="company_north_star.json")
    parser.add_argument("--feature_map", default="feature_goal_map.json")
    parser.add_argument("--output", default="segment_goals.csv")
    parser.add_argument(
        "--lifecycle",
        nargs="+",
        choices=["trial", "paid", "churned", "inactive"],
        default=None
    )

    args, unknown = parser.parse_known_args()
    if unknown:
        print("⚠️ Ignoring notebook/kernel arguments:", unknown)

    return run(
        segments_path=args.segments,
        north_star_path=args.north_star,
        feature_map_path=args.feature_map,
        output_path=args.output,
        lifecycle_filter=args.lifecycle,
    )


if __name__ == "__main__":
    main()

⚠️ Ignoring notebook/kernel arguments: ['-f', '/root/.local/share/jupyter/runtime/kernel-0af69c41-cf73-4608-b37d-bb78d5d72e8a.json']

🚀 Segment Goal Generator — Project Aurora

Current working directory: /content
Available files:
 - .config
 - .ipynb_checkpoints
 - allowed_tone_hook_matrix.json
 - cleaned_synthetic_behavioral_dataset_2000.xlsx
 - company_north_star.json
 - experiment_results.csv
 - feature_goal_map.json
 - iteration_0_before_learning
 - iteration_1_after_learning
 - outputs
 - sample_data
 - segment_goals.csv
 - speakx_knowledge_bank.md
 - user_segments_output.csv

✅ Loaded North Star file: company_north_star.json
✅ North Star metric: Monthly Active Learners (MAL) with 7+ Day Practice Streak
✅ Loaded Feature Map file: feature_goal_map.json
✅ Total feature-goal mappings: 0
✅ Loaded 2000 user rows from: user_segments_output.csv

Detected columns in CSV:
['user_id', 'lifecycle_stage', 'days_since_signup', 'age_band', 'region', 'sessions_last_7d', 'exercises_completed_7d',


⚠️ Gemini error for [SEG_00 x paid]: 404 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ListModels to see the list of available models and their supported methods.
✓
[ 2/10] SEG_00 (Moderately Active Users) x paid ... 


⚠️ Gemini error for [SEG_00 x paid]: 404 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ListModels to see the list of available models and their supported methods.
✓
[ 3/10] SEG_00 (Moderately Active Users) x paid ... 


⚠️ Gemini error for [SEG_00 x paid]: 404 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ListModels to see the list of available models and their supported methods.
✓
[ 4/10] SEG_00 (Moderately Active Users) x trial ... 


⚠️ Gemini error for [SEG_00 x trial]: 404 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ListModels to see the list of available models and their supported methods.
✓
[ 5/10] SEG_00 (Moderately Active Users) x trial ... 

...


⚠️ Gemini error for [SEG_01 x churned]: 404 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ListModels to see the list of available models and their supported methods.
✓
[ 7/10] SEG_01 (Low Activity Users) x inactive ... 


⚠️ Gemini error for [SEG_01 x inactive]: 404 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ListModels to see the list of available models and their supported methods.
✓
[ 8/10] SEG_01 (Low Activity Users) x trial ... 


⚠️ Gemini error for [SEG_01 x trial]: 404 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ListModels to see the list of available models and their supported methods.
✓
[ 9/10] SEG_01 (Low Activity Users) x trial ... 


⚠️ Gemini error for [SEG_01 x trial]: 404 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ListModels to see the list of available models and their supported methods.
✓
[10/10] SEG_01 (Low Activity Users) x trial ... 


⚠️ Gemini error for [SEG_01 x trial]: 404 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ListModels to see the list of available models and their supported methods.
✓

✅ Saved output to: segment_goals.csv
✅ Output rows: 10
✅ Output columns: ['segment_id', 'segment_name', 'lifecycle_stage', 'user_count_in_cohort', 'churn_risk', 'avg_activeness_score', 'top_octolysis_drive', 'recommended_notif_freq', 'primary_goal', 'sub_goal_1', 'sub_goal_2', 'sub_goal_3', 'recommended_features', 'north_star_lever', 'core_drive', 'success_metric', 'communication_priority', 'churn_risk_response_strategy', 'north_star_alignment_note', 'day', 'day_focus']

Preview:
segment_id            segment_name lifecycle_stage churn_risk day                               primary_goal day_focus
    SEG_00 Moderately Active Users    

In [ ]:
"""
Task 2: Communication & Timing Intelligence — Hybrid Version (Fixed)
====================================================================
Goal:
- Keep same output format as the original Task 2 pipeline
- Reduce unnecessary hardcoding
- Support random datasets with the same expected schema
- Use OpenAI for themes and message templates when available
- Keep LinUCB for timing intelligence
- Use only minimal emergency fallback when OpenAI is unavailable

Outputs:
  1. communication_themes.csv
  2. message_templates.csv
  3. timing_recommendations.csv
  4. user_notification_schedule.csv
"""

import os
import json
import re
import warnings
from pathlib import Path
from typing import Dict, List, Any

import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings("ignore")

# =============================================================================
# OPTIONAL OPENAI
# =============================================================================
OPENAI_AVAILABLE = False
try:
    from openai import OpenAI
    OPENAI_AVAILABLE = True
except Exception:
    OPENAI_AVAILABLE = False

OPENAI_MODEL = "gpt-4o-mini"
_openai_client = None


def get_openai_client():
    global _openai_client
    if not OPENAI_AVAILABLE:
        return None
    if _openai_client is None:
        api_key = os.environ.get("OPENAI_API_KEY", "").strip()
        if not api_key:
            return None
        _openai_client = OpenAI(api_key=api_key)
    return _openai_client


def is_openai_enabled() -> bool:
    return get_openai_client() is not None


# =============================================================================
# CONSTANTS
# =============================================================================
TIME_WINDOWS = {
    "early_morning":  {"range": "06:00-08:59", "use": "Morning motivation, habit trigger", "start": "07:00"},
    "mid_morning":    {"range": "09:00-11:59", "use": "Work break reminder", "start": "10:00"},
    "afternoon":      {"range": "12:00-14:59", "use": "Lunch break engagement", "start": "13:00"},
    "late_afternoon": {"range": "15:00-17:59", "use": "Productivity boost", "start": "16:00"},
    "evening":        {"range": "18:00-20:59", "use": "Post-work learning", "start": "19:00"},
    "night":          {"range": "21:00-23:59", "use": "End-of-day recap, streak save", "start": "22:00"},
}
WINDOW_NAMES = list(TIME_WINDOWS.keys())

FREQ_BANDS = [
    (0.7, float("inf"), 7, 9, "high_active", "Maximum engagement"),
    (0.4, 0.7,          5, 6, "moderate_active", "Balanced approach"),
    (0.0, 0.4,          3, 4, "low_active", "Conservative, avoid fatigue"),
]
GUARDRAIL_UNINSTALL_THRESHOLD = 0.02
GUARDRAIL_REDUCTION = 2

OCTOLYSIS_DRIVES = [
    "Epic Meaning",
    "Accomplishment",
    "Empowerment",
    "Ownership",
    "Social Influence",
    "Scarcity",
    "Unpredictability",
    "Loss Avoidance",
]

DEFAULT_ALLOWED_TONES = [
    "Empowerment",
    "Accomplishment",
    "Encouragement",
    "Social Influence",
    "Scarcity (Positive)",
    "Ownership",
]

PROGRESSION_STAGES = ["awareness", "engagement", "habit", "mastery", "retention"]

NOTEBOOK_CONFIG = {
    "segments": "user_segments.csv",
    "goals": "segment_goals.csv",
    "north_star": "company_north_star.json",
    "tone_matrix": "allowed_tone_hook_matrix.json",
    "feature_map": "feature_goal_map.json",
    "output_dir": "iteration_0_before_learning",
    "max_users_schedule": 2000,
}


# =============================================================================
# HELPERS
# =============================================================================
def _is_jupyter() -> bool:
    try:
        shell = get_ipython().__class__.__name__  # noqa
        return shell in ("ZMQInteractiveShell", "TerminalInteractiveShell", "google.colab._shell")
    except NameError:
        return False


def ensure_dir(path: str):
    Path(path).mkdir(parents=True, exist_ok=True)


def safe_read_json(path: str, default=None):
    if default is None:
        default = {}
    if not os.path.exists(path):
        print(f"⚠️ JSON file not found: {path}")
        return default
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception as e:
        print(f"⚠️ Failed to read JSON {path}: {e}")
        return default


def safe_read_csv(path: str, default=None):
    if default is None:
        default = pd.DataFrame()
    if not os.path.exists(path):
        print(f"⚠️ CSV file not found: {path}")
        return default
    try:
        return pd.read_csv(path)
    except Exception as e:
        print(f"⚠️ Failed to read CSV {path}: {e}")
        return default


def normalize_text_col(df: pd.DataFrame, col: str, lower: bool = False, default: str = "unknown"):
    if col not in df.columns:
        df[col] = default
    df[col] = df[col].fillna(default).astype(str).str.strip()
    if lower:
        df[col] = df[col].str.lower()
    return df


def normalize_numeric_col(df: pd.DataFrame, col: str, default: float = 0.0):
    if col not in df.columns:
        df[col] = default
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(default)
    return df


def clean_json_text(raw: str) -> str:
    if not raw:
        return ""
    raw = raw.strip()
    raw = re.sub(r"^```(?:json)?", "", raw, flags=re.IGNORECASE).strip()
    raw = re.sub(r"```$", "", raw).strip()
    return raw


def safe_json_loads(raw: str, default):
    try:
        return json.loads(clean_json_text(raw))
    except Exception:
        return default


def safe_choice(seq, fallback):
    return seq[0] if seq else fallback


# =============================================================================
# LOADERS
# =============================================================================
def load_segments(path: str) -> pd.DataFrame:
    df = safe_read_csv(path)
    if df.empty:
        raise FileNotFoundError(f"Segments file not found or empty: {path}")

    required_defaults = {
        "user_id": "",
        "cluster_id": 0,
        "segment_name": "Unknown Segment",
        "lifecycle_stage": "trial",
        "churn_risk": "low",
        "activeness_score": 0.3,
        "gamification_score": 0.0,
        "engagement_depth": 0.0,
        "notif_open_rate_30d": 0.2,
        "preferred_hour": 18,
        "sessions_last_7d": 0,
        "streak_current": 0,
        "motivation_score": 0.0,
        "days_since_signup": 0,
    }

    for col, default in required_defaults.items():
        if isinstance(default, str):
            normalize_text_col(df, col, lower=False, default=default)
        else:
            normalize_numeric_col(df, col, default=default)

    normalize_text_col(df, "lifecycle_stage", lower=True, default="trial")
    normalize_text_col(df, "churn_risk", lower=True, default="low")
    normalize_text_col(df, "segment_name", lower=False, default="Unknown Segment")

    df["cluster_id"] = pd.to_numeric(df["cluster_id"], errors="coerce").fillna(0).astype(int)
    df["preferred_hour"] = df["preferred_hour"].clip(0, 23)

    print(f"✅ Loaded {len(df)} users")
    print(f"✅ Segments found: {df['segment_name'].dropna().unique().tolist()}")
    return df


def load_goals(path: str) -> pd.DataFrame:
    if not os.path.exists(path):
        print("⚠️ segment_goals.csv not found — continuing without goal context.")
        return pd.DataFrame()

    df = safe_read_csv(path)
    if df.empty:
        print("⚠️ segment_goals.csv empty or unreadable — continuing without goal context.")
        return pd.DataFrame()

    normalize_text_col(df, "segment_name", lower=False, default="Unknown Segment")
    normalize_text_col(df, "lifecycle_stage", lower=True, default="trial")
    normalize_text_col(df, "primary_goal", lower=False, default="")

    if "primary_goal" in df.columns:
        df = df[~df["primary_goal"].fillna("").astype(str).str.startswith("ERROR")].copy()

    for col in ["sub_goal_1", "sub_goal_2", "north_star_lever", "core_drive"]:
        normalize_text_col(df, col, lower=False, default="")

    print(f"✅ Loaded {len(df)} goal rows")
    return df


# =============================================================================
# PROFILE AGGREGATION
# =============================================================================
def aggregate_segment_profiles(users_df: pd.DataFrame) -> pd.DataFrame:
    agg = (
        users_df.groupby(["cluster_id", "segment_name", "lifecycle_stage", "churn_risk"], dropna=False)
        .agg(
            user_count=("user_id", "count"),
            avg_activeness=("activeness_score", "mean"),
            avg_gamification=("gamification_score", "mean"),
            avg_engagement_depth=("engagement_depth", "mean"),
            avg_notif_open_rate=("notif_open_rate_30d", "mean"),
            avg_preferred_hour=("preferred_hour", "mean"),
            avg_sessions=("sessions_last_7d", "mean"),
            avg_streak=("streak_current", "mean"),
            avg_motivation=("motivation_score", "mean"),
        )
        .reset_index()
    )

    agg["segment_id"] = (
        "SEG_"
        + agg["cluster_id"].astype(int).astype(str).str.zfill(2)
        + "_"
        + agg["lifecycle_stage"].astype(str).str[:3].str.upper()
        + "_"
        + agg["churn_risk"].astype(str).str[:3].str.upper()
    )

    agg["top_octolysis"] = "Accomplishment"
    print(f"✅ Aggregated {len(agg)} segment × lifecycle profiles")
    return agg


# =============================================================================
# TIMING BANDIT
# =============================================================================
def _hour_to_window_idx(hour: float) -> int:
    h = int(round(float(hour))) % 24
    if 6 <= h <= 8:
        return 0
    if 9 <= h <= 11:
        return 1
    if 12 <= h <= 14:
        return 2
    if 15 <= h <= 17:
        return 3
    if 18 <= h <= 20:
        return 4
    if 21 <= h <= 23:
        return 5
    return 4


class LinUCB:
    def __init__(self, n_arms: int, n_features: int, alpha: float = 1.0):
        self.n_arms = n_arms
        self.n_features = n_features
        self.alpha = alpha
        self.A = [np.identity(n_features) for _ in range(n_arms)]
        self.b = [np.zeros(n_features) for _ in range(n_arms)]

    def select_arm(self, context: np.ndarray) -> int:
        x = context.reshape(-1)
        ucb_scores = []
        for a in range(self.n_arms):
            A_inv = np.linalg.inv(self.A[a])
            theta = A_inv @ self.b[a]
            exploit = theta @ x
            explore = self.alpha * np.sqrt(x @ A_inv @ x)
            ucb_scores.append(exploit + explore)
        return int(np.argmax(ucb_scores))

    def update(self, arm: int, context: np.ndarray, reward: float):
        x = context.reshape(-1)
        self.A[arm] += np.outer(x, x)
        self.b[arm] += reward * x

    def get_arm_probabilities(self, context: np.ndarray) -> np.ndarray:
        x = context.reshape(-1)
        scores = []
        for a in range(self.n_arms):
            A_inv = np.linalg.inv(self.A[a])
            theta = A_inv @ self.b[a]
            scores.append(theta @ x + self.alpha * np.sqrt(x @ A_inv @ x))
        scores = np.array(scores, dtype=float)
        e = np.exp(scores - scores.max())
        return e / e.sum()


def build_context_vector(row: dict, lifecycle_enc: LabelEncoder, churn_enc: LabelEncoder) -> np.ndarray:
    lifecycle = str(row.get("lifecycle_stage", "trial"))
    churn = str(row.get("churn_risk", "low"))

    lc_val = lifecycle_enc.transform([lifecycle])[0] / max(len(lifecycle_enc.classes_) - 1, 1)
    cr_val = churn_enc.transform([churn])[0] / max(len(churn_enc.classes_) - 1, 1)

    return np.array([
        float(row.get("avg_activeness", 0.0)),
        float(row.get("avg_notif_open_rate", 0.0)),
        float(row.get("avg_preferred_hour", 12.0)) / 23.0,
        lc_val,
        cr_val,
    ], dtype=float)


def simulate_reward(arm_idx: int, row: dict) -> float:
    preferred_window = _hour_to_window_idx(float(row.get("avg_preferred_hour", 18)))
    distance = abs(arm_idx - preferred_window)
    proximity_bonus = max(0.0, 1.0 - distance * 0.25)

    base_reward = float(row.get("avg_notif_open_rate", 0.2))
    churn_penalty = {
        "low": 0.0,
        "medium": 0.05,
        "high": 0.10,
        "critical": 0.20,
    }.get(str(row.get("churn_risk", "low")).lower(), 0.05)

    return float(min(1.0, max(0.0, base_reward * (1 + proximity_bonus) - churn_penalty)))


def run_bandit_training(profiles_df: pd.DataFrame, n_rounds: int = 500, alpha: float = 1.2):
    if profiles_df.empty:
        raise ValueError("profiles_df is empty. Cannot train bandit.")

    lc_enc = LabelEncoder().fit(profiles_df["lifecycle_stage"].astype(str).unique())
    cr_enc = LabelEncoder().fit(profiles_df["churn_risk"].astype(str).unique())

    bandit = LinUCB(n_arms=len(WINDOW_NAMES), n_features=5, alpha=alpha)
    rng = np.random.default_rng(42)

    for _ in range(n_rounds):
        row = profiles_df.iloc[int(rng.integers(0, len(profiles_df)))].to_dict()
        ctx = build_context_vector(row, lc_enc, cr_enc)
        arm = bandit.select_arm(ctx)
        reward = simulate_reward(arm, row)
        bandit.update(arm, ctx, reward)

    print(f"✅ LinUCB trained: {n_rounds} rounds")
    return bandit, lc_enc, cr_enc


def generate_timing_recommendations(profiles_df: pd.DataFrame, bandit: LinUCB, lc_enc: LabelEncoder, cr_enc: LabelEncoder) -> pd.DataFrame:
    rows = []
    for _, profile in profiles_df.iterrows():
        ctx = build_context_vector(profile.to_dict(), lc_enc, cr_enc)
        probs = bandit.get_arm_probabilities(ctx)
        ranked = np.argsort(probs)[::-1]

        primary_idx = int(ranked[0])
        secondary_idx = int(ranked[1]) if len(ranked) > 1 else int(ranked[0])

        rows.append({
            "segment_id": profile["segment_id"],
            "segment_name": profile["segment_name"],
            "lifecycle_stage": profile["lifecycle_stage"],
            "churn_risk": profile["churn_risk"],
            "avg_activeness": round(float(profile["avg_activeness"]), 3),
            "avg_notif_open_rate": round(float(profile["avg_notif_open_rate"]), 3),
            "primary_window": WINDOW_NAMES[primary_idx],
            "primary_window_range": TIME_WINDOWS[WINDOW_NAMES[primary_idx]]["range"],
            "primary_window_prob": round(float(probs[primary_idx]), 4),
            "expected_ctr_primary": round(float(profile["avg_notif_open_rate"]) * float(probs[primary_idx]) * 2, 4),
            "secondary_window": WINDOW_NAMES[secondary_idx],
            "secondary_window_range": TIME_WINDOWS[WINDOW_NAMES[secondary_idx]]["range"],
            "secondary_window_prob": round(float(probs[secondary_idx]), 4),
            "expected_ctr_secondary": round(float(profile["avg_notif_open_rate"]) * float(probs[secondary_idx]) * 2, 4),
            "bandit_algorithm": "LinUCB_disjoint",
            "context_features": "activeness,notif_open_rate,preferred_hour,lifecycle,churn_risk",
            "n_training_rounds": 500,
        })

    df = pd.DataFrame(rows)
    print(f"✅ Generated {len(df)} timing recommendation rows")
    return df


# =============================================================================
# FREQUENCY
# =============================================================================
def get_frequency_band(activeness: float, uninstall_rate: float = 0.0) -> dict:
    activeness = float(activeness)
    for lo, hi, min_f, max_f, label, strategy in FREQ_BANDS:
        if lo <= activeness < hi or (hi == float("inf") and activeness >= lo):
            if uninstall_rate > GUARDRAIL_UNINSTALL_THRESHOLD:
                min_f = max(1, min_f - GUARDRAIL_REDUCTION)
                max_f = max(1, max_f - GUARDRAIL_REDUCTION)
                strategy += " [GUARDRAIL APPLIED]"
            recommended = (min_f + max_f) // 2
            return {
                "freq_band": label,
                "min_notifs": min_f,
                "max_notifs": max_f,
                "recommended": recommended,
                "strategy": strategy,
            }
    return {
        "freq_band": "low_active",
        "min_notifs": 3,
        "max_notifs": 4,
        "recommended": 3,
        "strategy": "Conservative",
    }


# =============================================================================
# THEME GENERATION
# =============================================================================
THEME_SYSTEM_PROMPT = """
You are a communication strategy expert for a learning or engagement product.

Return ONLY valid JSON.
Use only these Octolysis drives:
Epic Meaning, Accomplishment, Empowerment, Ownership,
Social Influence, Scarcity, Unpredictability, Loss Avoidance

Allowed tones:
Empowerment, Accomplishment, Encouragement,
Social Influence, Scarcity (Positive), Ownership
"""


def infer_theme_from_profile(profile: dict, tone_matrix: dict) -> dict:
    allowed_tones = [t.get("tone", "") for t in tone_matrix.get("allowed_tones", []) if isinstance(t, dict)]
    if not allowed_tones:
        allowed_tones = DEFAULT_ALLOWED_TONES

    activeness = float(profile.get("avg_activeness", 0))
    open_rate = float(profile.get("avg_notif_open_rate", 0))
    streak = float(profile.get("avg_streak", 0))
    motivation = float(profile.get("avg_motivation", 0))
    churn = str(profile.get("churn_risk", "low")).lower()

    if churn in ["high", "critical"]:
        primary = "Loss Avoidance"
        secondary = "Empowerment"
        primary_tone = "Encouragement"
    elif streak >= 5:
        primary = "Accomplishment"
        secondary = "Ownership"
        primary_tone = "Accomplishment"
    elif activeness >= 0.65 or motivation >= 0.65:
        primary = "Empowerment"
        secondary = "Accomplishment"
        primary_tone = "Empowerment"
    elif open_rate >= 0.30:
        primary = "Social Influence"
        secondary = "Empowerment"
        primary_tone = "Social Influence"
    else:
        primary = "Empowerment"
        secondary = "Ownership"
        primary_tone = "Encouragement"

    secondary_tone = "Empowerment" if "Empowerment" in allowed_tones else safe_choice(allowed_tones, "Encouragement")
    if primary_tone not in allowed_tones:
        primary_tone = safe_choice(allowed_tones, "Encouragement")

    return {
        "segment_id": profile["segment_id"],
        "segment_name": profile["segment_name"],
        "lifecycle_stage": profile["lifecycle_stage"],
        "primary_theme": primary,
        "secondary_theme": secondary if secondary in OCTOLYSIS_DRIVES else "Empowerment",
        "tertiary_theme": "Ownership",
        "primary_tone": primary_tone,
        "secondary_tone": secondary_tone,
        "theme_rationale": "Generated from segment-level behavioral signals.",
        "primary_hook": "Build momentum with one step today",
        "secondary_hook": "Consistent effort creates visible progress",
        "avoided_tones": ["Fear/Anxiety", "Shame/Guilt"],
        "north_star_alignment": "Supports recurring engagement and sustained progress.",
        "churn_risk_adaptation": f"Adjusted for churn label {profile.get('churn_risk', 'low')}.",
    }


def generate_communication_theme(profile: dict, north_star: dict, tone_matrix: dict, goals_df: pd.DataFrame) -> dict:
    if not is_openai_enabled():
        return infer_theme_from_profile(profile, tone_matrix)

    allowed_tones = [t.get("tone", "") for t in tone_matrix.get("allowed_tones", []) if isinstance(t, dict)]
    hooks = tone_matrix.get("hook_taxonomy", [])
    if not allowed_tones:
        allowed_tones = DEFAULT_ALLOWED_TONES

    goal_context = ""
    if not goals_df.empty:
        match = goals_df[
            (goals_df["segment_name"].astype(str).str.lower() == str(profile["segment_name"]).lower()) &
            (goals_df["lifecycle_stage"].astype(str).str.lower() == str(profile["lifecycle_stage"]).lower())
        ]
        if not match.empty:
            row = match.iloc[0]
            goal_context = f"""
Segment Goals:
primary_goal: {row.get('primary_goal', '')}
sub_goal_1: {row.get('sub_goal_1', '')}
sub_goal_2: {row.get('sub_goal_2', '')}
north_star_lever: {row.get('north_star_lever', '')}
core_drive: {row.get('core_drive', '')}
"""

    schema = {
        "segment_id": "<string>",
        "segment_name": "<string>",
        "lifecycle_stage": "<string>",
        "primary_theme": "<one of 8 Octolysis drives>",
        "secondary_theme": "<one of 8 Octolysis drives>",
        "tertiary_theme": "<one of 8 Octolysis drives>",
        "primary_tone": "<allowed tone>",
        "secondary_tone": "<allowed tone>",
        "theme_rationale": "<short text>",
        "primary_hook": "<short hook>",
        "secondary_hook": "<short hook>",
        "avoided_tones": ["<tone1>", "<tone2>"],
        "north_star_alignment": "<short text>",
        "churn_risk_adaptation": "<short text>",
    }

    prompt = f"""
North Star: {north_star.get('north_star_metric', '')}
Proxy Formula: {north_star.get('measurable_proxy', {}).get('formula', '')}

Segment Profile:
segment_id: {profile['segment_id']}
segment_name: {profile['segment_name']}
lifecycle: {profile['lifecycle_stage']}
churn_risk: {profile['churn_risk']}
avg_activeness: {round(float(profile.get('avg_activeness', 0)), 3)}
avg_sessions_7d: {round(float(profile.get('avg_sessions', 0)), 2)}
avg_streak: {round(float(profile.get('avg_streak', 0)), 2)}
avg_motivation: {round(float(profile.get('avg_motivation', 0)), 3)}

{goal_context}

Allowed tones: {allowed_tones}
Example hooks: {[h.get('hook', '') for h in hooks[:6] if isinstance(h, dict)]}

Return JSON only with this schema:
{json.dumps(schema, indent=2)}
"""

    try:
        client = get_openai_client()
        response = client.chat.completions.create(
            model=OPENAI_MODEL,
            temperature=0.7,
            max_tokens=600,
            messages=[
                {"role": "system", "content": THEME_SYSTEM_PROMPT},
                {"role": "user", "content": prompt},
            ],
        )
        raw = response.choices[0].message.content.strip()
        result = safe_json_loads(raw, default={})
        if not result:
            return infer_theme_from_profile(profile, tone_matrix)

        result.setdefault("segment_id", profile["segment_id"])
        result.setdefault("segment_name", profile["segment_name"])
        result.setdefault("lifecycle_stage", profile["lifecycle_stage"])
        result.setdefault("primary_theme", "Accomplishment")
        result.setdefault("secondary_theme", "Empowerment")
        result.setdefault("tertiary_theme", "Ownership")
        result.setdefault("primary_tone", "Encouragement")
        result.setdefault("secondary_tone", "Empowerment")
        result.setdefault("theme_rationale", "Generated theme.")
        result.setdefault("primary_hook", "Keep moving forward today")
        result.setdefault("secondary_hook", "Every session builds confidence")
        result.setdefault("avoided_tones", ["Fear/Anxiety", "Shame/Guilt"])
        result.setdefault("north_star_alignment", "Supports consistent practice.")
        result.setdefault("churn_risk_adaptation", "Adjusted to churn label.")
        return result
    except Exception:
        return infer_theme_from_profile(profile, tone_matrix)


def generate_all_themes(profiles_df: pd.DataFrame, north_star: dict, tone_matrix: dict, goals_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    total = len(profiles_df)
    for i, (_, profile) in enumerate(profiles_df.iterrows(), 1):
        print(f"  [{i:>2}/{total}] Theme: {profile['segment_id']} ... ", end="")
        rows.append(generate_communication_theme(profile.to_dict(), north_star, tone_matrix, goals_df))
        print("✓")
    return pd.DataFrame(rows)


# =============================================================================
# TEMPLATE GENERATION — FIXED
# =============================================================================
TEMPLATE_SYSTEM_PROMPT = """
You are a bilingual (Hindi + English) notification copywriter for a learning app.

Return ONLY valid JSON.

Rules:
1. Generate exactly 5 templates.
2. Each template must include:
   - en_title
   - en_body
   - hi_title
   - hi_body
3. Hindi must be natural Devanagari and should NOT just mirror the same sentence every time.
4. Make templates clearly different across progression stages:
   awareness → engagement → habit → mastery → retention
5. Respect the given theme, tone, feature, goal, and churn context.
6. Avoid fear, shame, guilt, manipulation, and negative pressure.
7. Keep:
   - en_title <= 40 chars
   - en_body <= 80 chars
   - hi_title <= 40 chars
   - hi_body <= 100 chars
8. Titles and bodies should reflect the specific tone. Tone must visibly affect wording.
9. Output JSON only, no markdown.
"""


def infer_relevant_features(feature_map: list, profile: dict) -> List[dict]:
    if not isinstance(feature_map, list) or len(feature_map) == 0:
        return []

    lifecycle = str(profile.get("lifecycle_stage", "")).lower().strip()
    filtered = []
    generic = []

    for item in feature_map:
        if not isinstance(item, dict):
            continue
        item_stage = str(item.get("lifecycle_stage", "")).lower().strip()
        if item_stage == lifecycle:
            filtered.append(item)
        elif item_stage == "":
            generic.append(item)

    if not filtered:
        filtered = generic if generic else [x for x in feature_map if isinstance(x, dict)]

    return filtered[:4]


def _tone_copy_bank(tone: str) -> Dict[str, List[str]]:
    tone = str(tone).strip().lower()

    banks = {
        "empowerment": {
            "en_title": ["Take charge", "Your next step", "Move forward", "Build your skill", "You can do this"],
            "en_body": [
                "Use {feature} to build confidence in {goal}.",
                "Take one strong step with {feature} toward {goal}.",
                "Your progress in {goal} can grow with {feature}.",
            ],
            "hi_title": ["अपना अगला कदम", "आगे बढ़ें", "आप कर सकते हैं", "अपनी ताकत बढ़ाएं", "नई शुरुआत करें"],
            "hi_body": [
                "{feature} के साथ {goal} में आत्मविश्वास बढ़ाएं।",
                "{feature} का उपयोग करके {goal} की ओर आगे बढ़ें।",
                "{feature} आपकी {goal} यात्रा को मजबूत बना सकता है।",
            ],
        },
        "accomplishment": {
            "en_title": ["See your progress", "Keep improving", "Reach your milestone", "Make today count", "Grow your results"],
            "en_body": [
                "Use {feature} to improve your results in {goal}.",
                "One more step with {feature} can strengthen {goal}.",
                "Build visible progress in {goal} through {feature}.",
            ],
            "hi_title": ["अपनी प्रगति देखें", "आज और बेहतर करें", "अपना लक्ष्य बढ़ाएं", "एक कदम और", "अपना सुधार जारी रखें"],
            "hi_body": [
                "{feature} के साथ {goal} में बेहतर परिणाम पाएं।",
                "{feature} का एक और उपयोग {goal} को मजबूत बना सकता है।",
                "{feature} से {goal} में स्पष्ट प्रगति बनाएं।",
            ],
        },
        "encouragement": {
            "en_title": ["Keep going", "A small step today", "Stay on track", "You’re doing well", "Keep the flow"],
            "en_body": [
                "A small step with {feature} can support {goal}.",
                "Stay steady with {feature} and continue {goal}.",
                "Keep your momentum going through {feature}.",
            ],
            "hi_title": ["जारी रखें", "आज एक छोटा कदम", "रफ्तार बनाए रखें", "आप अच्छा कर रहे हैं", "आगे बढ़ते रहें"],
            "hi_body": [
                "{feature} के साथ एक छोटा कदम {goal} में मदद कर सकता है।",
                "{feature} का नियमित उपयोग करके {goal} जारी रखें।",
                "{feature} आपकी प्रगति की रफ्तार बनाए रख सकता है।",
            ],
        },
        "social influence": {
            "en_title": ["Keep your momentum", "Show your growth", "Stay visible", "Build learning momentum", "Keep improving"],
            "en_body": [
                "Use {feature} to make your {goal} progress more visible.",
                "Keep learning momentum strong with {feature}.",
                "Your consistency in {goal} can stand out through {feature}.",
            ],
            "hi_title": ["अपनी प्रगति दिखाएं", "रफ्तार बनाए रखें", "अपनी बढ़त बढ़ाएं", "सीखना जारी रखें", "नियमित रहें"],
            "hi_body": [
                "{feature} से {goal} की प्रगति को और स्पष्ट बनाएं।",
                "{feature} के साथ अपनी सीखने की रफ्तार मजबूत रखें।",
                "{feature} आपकी {goal} में निरंतरता को बेहतर दिखा सकता है।",
            ],
        },
        "scarcity (positive)": {
            "en_title": ["Use today well", "Don’t miss this step", "Make this chance count", "Use this moment", "Take this chance"],
            "en_body": [
                "This is a good moment to use {feature} for {goal}.",
                "Use today’s opportunity with {feature} to support {goal}.",
                "A timely step with {feature} can help {goal}.",
            ],
            "hi_title": ["आज का मौका लें", "यह कदम न छोड़ें", "अभी शुरू करें", "इस अवसर का उपयोग करें", "आज आगे बढ़ें"],
            "hi_body": [
                "{feature} के साथ {goal} पर काम करने का यह अच्छा समय है।",
                "{feature} का आज उपयोग करके {goal} को आगे बढ़ाएं।",
                "{feature} के साथ सही समय पर लिया गया कदम {goal} में मदद करेगा।",
            ],
        },
        "ownership": {
            "en_title": ["Own your routine", "Keep your practice alive", "Stay in your flow", "Build your routine", "Continue your path"],
            "en_body": [
                "Use {feature} to strengthen your routine around {goal}.",
                "Keep your practice steady with {feature}.",
                "Make {feature} part of your {goal} routine.",
            ],
            "hi_title": ["अपनी आदत बनाए रखें", "अपनी दिनचर्या मजबूत करें", "नियमित रहें", "अभ्यास जारी रखें", "अपनी रफ्तार रखें"],
            "hi_body": [
                "{feature} के साथ {goal} की अपनी दिनचर्या मजबूत करें।",
                "{feature} का उपयोग करके अपना अभ्यास नियमित रखें।",
                "{feature} को {goal} की आदत का हिस्सा बनाएं।",
            ],
        },
    }

    return banks.get(tone, banks["encouragement"])


def minimal_emergency_templates(profile: dict, theme: dict, relevant_features: List[dict], template_idx_offset: int = 0) -> List[dict]:
    """
    Fallback only if OpenAI is unavailable or response parsing fails.
    Dynamic and tone-aware, not same Hindi every time.
    """
    feature_name = relevant_features[0].get("feature", "core feature") if relevant_features else "core feature"
    goal_name = relevant_features[0].get("user_goal", "daily progress") if relevant_features else "daily progress"
    tone = theme.get("primary_tone", "Encouragement")
    tone_bank = _tone_copy_bank(tone)

    stage_body_suffix_en = {
        "awareness": "Start by exploring one simple step.",
        "engagement": "Try one action and see your progress.",
        "habit": "Repeat it today to build consistency.",
        "mastery": "Go deeper and improve your quality.",
        "retention": "Keep the momentum going today.",
    }

    stage_body_suffix_hi = {
        "awareness": "शुरुआत एक छोटे कदम से करें।",
        "engagement": "एक बार आजमाकर अपनी प्रगति देखें।",
        "habit": "इसे आज दोहराकर निरंतरता बनाएं।",
        "mastery": "और बेहतर करने के लिए गहराई से करें।",
        "retention": "अपनी रफ्तार आज भी बनाए रखें।",
    }

    out = []
    for i, stage in enumerate(PROGRESSION_STAGES):
        en_title = tone_bank["en_title"][i % len(tone_bank["en_title"])]
        en_body_core = tone_bank["en_body"][i % len(tone_bank["en_body"])].format(
            feature=feature_name, goal=goal_name
        )
        hi_title = tone_bank["hi_title"][i % len(tone_bank["hi_title"])]
        hi_body_core = tone_bank["hi_body"][i % len(tone_bank["hi_body"])].format(
            feature=feature_name, goal=goal_name
        )

        out.append({
            "template_id": f"TMPL_{str(template_idx_offset + i + 1).zfill(4)}",
            "segment_id": profile["segment_id"],
            "lifecycle_stage": profile["lifecycle_stage"],
            "theme": theme.get("primary_theme", "Accomplishment"),
            "goal": goal_name,
            "tone": tone,
            "hook_type": theme.get("primary_theme", "Accomplishment"),
            "feature_reference": feature_name,
            "progression_stage": stage,
            "en_title": en_title[:40],
            "en_body": f"{en_body_core} {stage_body_suffix_en[stage]}"[:80],
            "hi_title": hi_title[:40],
            "hi_body": f"{hi_body_core} {stage_body_suffix_hi[stage]}"[:100],
            "channel": "push",
            "cta": "Continue",
        })
    return out


def generate_message_templates(profile: dict, theme: dict, tone_matrix: dict, feature_map: list, template_idx_offset: int = 0) -> List[dict]:
    relevant_features = infer_relevant_features(feature_map, profile)

    allowed_tones = [
        t.get("tone", "") for t in tone_matrix.get("allowed_tones", [])
        if isinstance(t, dict)
    ]
    if not allowed_tones:
        allowed_tones = DEFAULT_ALLOWED_TONES

    feature_context = []
    for feat in relevant_features:
        if not isinstance(feat, dict):
            continue
        feature_context.append({
            "feature": feat.get("feature", ""),
            "user_goal": feat.get("user_goal", ""),
            "outcome": feat.get("outcome", ""),
            "success_metric": feat.get("success_metric", ""),
            "lifecycle_stage": feat.get("lifecycle_stage", ""),
        })

    schema = {
        "templates": [
            {
                "template_id": f"TMPL_{str(template_idx_offset + i + 1).zfill(4)}",
                "segment_id": profile["segment_id"],
                "lifecycle_stage": profile["lifecycle_stage"],
                "theme": theme.get("primary_theme", "Accomplishment"),
                "goal": "<goal this template addresses>",
                "tone": "<one allowed tone>",
                "hook_type": "<theme / motivational hook>",
                "feature_reference": "<feature referenced>",
                "progression_stage": "<awareness|engagement|habit|mastery|retention>",
                "en_title": "<English title max 40 chars>",
                "en_body": "<English body max 80 chars>",
                "hi_title": "<Hindi title in Devanagari max 40 chars>",
                "hi_body": "<Hindi body in Devanagari max 100 chars>",
                "channel": "<push|in_app|email>",
                "cta": "<CTA max 20 chars>"
            }
            for i in range(5)
        ]
    }

    prompt = f"""
Generate exactly 5 bilingual notification templates.

Segment profile:
- segment_id: {profile.get('segment_id', '')}
- segment_name: {profile.get('segment_name', '')}
- lifecycle_stage: {profile.get('lifecycle_stage', '')}
- churn_risk: {profile.get('churn_risk', '')}
- avg_activeness: {round(float(profile.get('avg_activeness', 0.0)), 3)}
- avg_notif_open_rate: {round(float(profile.get('avg_notif_open_rate', 0.0)), 3)}
- avg_sessions: {round(float(profile.get('avg_sessions', 0.0)), 3)}
- avg_streak: {round(float(profile.get('avg_streak', 0.0)), 3)}
- avg_motivation: {round(float(profile.get('avg_motivation', 0.0)), 3)}

Theme context:
- primary_theme: {theme.get('primary_theme', 'Accomplishment')}
- secondary_theme: {theme.get('secondary_theme', 'Empowerment')}
- tertiary_theme: {theme.get('tertiary_theme', 'Ownership')}
- primary_tone: {theme.get('primary_tone', 'Encouragement')}
- secondary_tone: {theme.get('secondary_tone', 'Empowerment')}
- primary_hook: {theme.get('primary_hook', '')}
- secondary_hook: {theme.get('secondary_hook', '')}
- theme_rationale: {theme.get('theme_rationale', '')}

Allowed tones:
{allowed_tones}

Relevant feature / goal context:
{json.dumps(feature_context, ensure_ascii=False, indent=2)}

Required progression order:
1. awareness
2. engagement
3. habit
4. mastery
5. retention

Important constraints:
- Make the 5 templates meaningfully different from one another.
- Hindi title and Hindi body must vary across templates.
- Tone must be visible in wording.
- Use the feature and goal context wherever relevant.
- Do not invent aggressive or fear-based messaging.
- Keep titles and bodies short enough for notifications.
- Hindi must be natural and easy to understand.
- Return only JSON using exactly this schema:

{json.dumps(schema, ensure_ascii=False, indent=2)}
"""

    if not is_openai_enabled():
        return minimal_emergency_templates(profile, theme, relevant_features, template_idx_offset)

    try:
        client = get_openai_client()
        response = client.chat.completions.create(
            model=OPENAI_MODEL,
            temperature=0.8,
            max_tokens=1800,
            messages=[
                {"role": "system", "content": TEMPLATE_SYSTEM_PROMPT},
                {"role": "user", "content": prompt},
            ],
        )

        raw = response.choices[0].message.content.strip()
        parsed = safe_json_loads(raw, default={})
        templates = parsed.get("templates", [])

        if not isinstance(templates, list) or len(templates) == 0:
            return minimal_emergency_templates(profile, theme, relevant_features, template_idx_offset)

        fallback = minimal_emergency_templates(profile, theme, relevant_features, template_idx_offset)

        cleaned = []
        for i in range(5):
            t = templates[i] if i < len(templates) and isinstance(templates[i], dict) else {}
            fb = fallback[i]

            cleaned.append({
                "template_id": t.get("template_id", fb["template_id"]),
                "segment_id": profile["segment_id"],
                "lifecycle_stage": profile["lifecycle_stage"],
                "theme": t.get("theme", fb["theme"]),
                "goal": t.get("goal", fb["goal"]),
                "tone": t.get("tone", fb["tone"]),
                "hook_type": t.get("hook_type", fb["hook_type"]),
                "feature_reference": t.get("feature_reference", fb["feature_reference"]),
                "progression_stage": t.get("progression_stage", fb["progression_stage"]),
                "en_title": str(t.get("en_title", fb["en_title"]))[:40],
                "en_body": str(t.get("en_body", fb["en_body"]))[:80],
                "hi_title": str(t.get("hi_title", fb["hi_title"]))[:40],
                "hi_body": str(t.get("hi_body", fb["hi_body"]))[:100],
                "channel": t.get("channel", fb["channel"]),
                "cta": str(t.get("cta", fb["cta"]))[:20],
            })

        return cleaned

    except Exception:
        return minimal_emergency_templates(profile, theme, relevant_features, template_idx_offset)


def generate_all_templates(profiles_df: pd.DataFrame, themes_df: pd.DataFrame, tone_matrix: dict, feature_map: list) -> pd.DataFrame:
    all_templates = []
    offset = 0
    total = len(profiles_df)

    print(f"✅ Generating templates for {total} segment profiles")

    for i, (_, profile) in enumerate(profiles_df.iterrows(), 1):
        sid = profile["segment_id"]
        print(f"  [{i:>2}/{total}] Templates: {sid} ... ", end="")
        match = themes_df[themes_df["segment_id"] == sid]
        theme = match.iloc[0].to_dict() if not match.empty else {
            "primary_theme": "Accomplishment",
            "secondary_theme": "Empowerment",
            "tertiary_theme": "Ownership",
            "primary_tone": "Encouragement",
            "secondary_tone": "Empowerment",
            "primary_hook": "Keep moving forward",
            "secondary_hook": "Progress grows with consistency",
            "theme_rationale": "Fallback theme"
        }
        templates = generate_message_templates(profile.to_dict(), theme, tone_matrix, feature_map, offset)
        all_templates.extend(templates)
        offset += 5
        print(f"✓ ({len(templates)} templates)")

    return pd.DataFrame(all_templates)


# =============================================================================
# USER SCHEDULE
# =============================================================================
def get_window_start_time(window_name: str) -> str:
    starts = {
        "early_morning": "07:00",
        "mid_morning": "10:00",
        "afternoon": "13:00",
        "late_afternoon": "16:00",
        "evening": "19:00",
        "night": "22:00",
    }
    return starts.get(window_name, "10:00")


def build_user_schedule(users_df: pd.DataFrame, profiles_df: pd.DataFrame, timing_df: pd.DataFrame, templates_df: pd.DataFrame, max_users: int = 2000) -> pd.DataFrame:
    timing_lookup = timing_df.set_index("segment_id").to_dict("index") if not timing_df.empty else {}
    template_lookup = templates_df.groupby("segment_id")["template_id"].apply(list).to_dict() if not templates_df.empty else {}
    channel_lookup = templates_df.set_index("template_id")["channel"].to_dict() if not templates_df.empty else {}

    profile_key = profiles_df.set_index(["cluster_id", "lifecycle_stage", "churn_risk"])["segment_id"].to_dict()

    schedule_rows = []
    rng = np.random.default_rng(42)

    for _, user in users_df.head(max_users).iterrows():
        key = (
            int(user["cluster_id"]),
            str(user["lifecycle_stage"]).lower(),
            str(user["churn_risk"]).lower(),
        )
        seg_id = profile_key.get(key)
        if seg_id is None:
            continue

        timing = timing_lookup.get(seg_id, {})
        tmpl_ids = template_lookup.get(seg_id, [])
        if not tmpl_ids:
            continue

        freq_config = get_frequency_band(float(user.get("activeness_score", 0.3)))
        n_notifs = min(int(freq_config["recommended"]), 9)
        lifecycle_day = int(pd.to_numeric(user.get("days_since_signup", 0), errors="coerce") if pd.notna(user.get("days_since_signup", 0)) else 0)

        row = {
            "user_id": user["user_id"],
            "segment_id": seg_id,
            "segment_name": user["segment_name"],
            "lifecycle_stage": user["lifecycle_stage"],
            "lifecycle_day": lifecycle_day,
            "churn_risk": user["churn_risk"],
            "activeness_score": round(float(user.get("activeness_score", 0.0)), 3),
            "freq_band": freq_config["freq_band"],
            "n_notifs_today": n_notifs,
            "primary_window": timing.get("primary_window", "evening"),
            "secondary_window": timing.get("secondary_window", "night"),
        }

        primary_time = get_window_start_time(timing.get("primary_window", "evening"))
        secondary_time = get_window_start_time(timing.get("secondary_window", "night"))

        for slot in range(1, 10):
            if slot <= n_notifs:
                tmpl = tmpl_ids[int(rng.integers(0, len(tmpl_ids)))]
                time_val = primary_time if slot % 2 == 1 else secondary_time
                ch = channel_lookup.get(tmpl, "push")

                row[f"notif_{slot}_template_id"] = tmpl
                row[f"notif_{slot}_time"] = time_val
                row[f"notif_{slot}_channel"] = ch
            else:
                row[f"notif_{slot}_template_id"] = ""
                row[f"notif_{slot}_time"] = ""
                row[f"notif_{slot}_channel"] = ""

        schedule_rows.append(row)

    df = pd.DataFrame(schedule_rows)
    print(f"✅ Generated schedule for {len(df)} users")
    return df


# =============================================================================
# MAIN RUN PIPELINE
# =============================================================================
def run(
    segments_path: str,
    goals_path: str,
    north_star_path: str,
    tone_matrix_path: str,
    feature_map_path: str,
    output_dir: str,
    max_users_schedule: int = 2000,
):
    ensure_dir(output_dir)

    print("\n" + "═" * 70)
    print("PROJECT AURORA — Task 2: Communication & Timing Intelligence")
    print("═" * 70)

    print("\n[1/6] Loading inputs...")
    users_df = load_segments(segments_path)
    goals_df = load_goals(goals_path)
    north_star = safe_read_json(north_star_path, default={})
    tone_matrix = safe_read_json(tone_matrix_path, default={})
    feat_map_raw = safe_read_json(feature_map_path, default=[])

    if isinstance(feat_map_raw, dict):
        feature_map = feat_map_raw.get("feature_goal_outcome_map", [])
    elif isinstance(feat_map_raw, list):
        feature_map = feat_map_raw
    else:
        feature_map = []

    print("\n[2/6] Aggregating segment × lifecycle profiles...")
    profiles_df = aggregate_segment_profiles(users_df)

    print("\n[3/6] Training LinUCB contextual bandit...")
    bandit, lc_enc, cr_enc = run_bandit_training(profiles_df, n_rounds=500, alpha=1.2)
    timing_df = generate_timing_recommendations(profiles_df, bandit, lc_enc, cr_enc)
    timing_path = os.path.join(output_dir, "timing_recommendations.csv")
    timing_df.to_csv(timing_path, index=False)
    print(f"   → Saved: {timing_path}")

    print("\n[4/6] Generating communication themes...")
    if is_openai_enabled():
        print("   OpenAI mode: ENABLED")
    else:
        print("   OpenAI mode: DISABLED — using dynamic fallback theme generation")
    themes_df = generate_all_themes(profiles_df, north_star, tone_matrix, goals_df)
    themes_path = os.path.join(output_dir, "communication_themes.csv")
    themes_df.to_csv(themes_path, index=False)
    print(f"   → Saved: {themes_path}")

    print("\n[5/6] Generating bilingual message templates...")
    if is_openai_enabled():
        print("   OpenAI mode: ENABLED")
    else:
        print("   OpenAI mode: DISABLED — using minimal emergency fallback templates")
    templates_df = generate_all_templates(profiles_df, themes_df, tone_matrix, feature_map)
    templates_path = os.path.join(output_dir, "message_templates.csv")
    templates_df.to_csv(templates_path, index=False)
    print(f"   → Saved: {templates_path}")

    print("\n[6/6] Building user notification schedules...")
    schedule_df = build_user_schedule(
        users_df=users_df,
        profiles_df=profiles_df,
        timing_df=timing_df,
        templates_df=templates_df,
        max_users=max_users_schedule,
    )
    schedule_path = os.path.join(output_dir, "user_notification_schedule.csv")
    schedule_df.to_csv(schedule_path, index=False)
    print(f"   → Saved: {schedule_path}")

    print("\n" + "═" * 70)
    print("TASK 2 COMPLETE")
    print(f"✓ timing_recommendations.csv       {len(timing_df)} rows")
    print(f"✓ communication_themes.csv        {len(themes_df)} rows")
    print(f"✓ message_templates.csv           {len(templates_df)} rows")
    print(f"✓ user_notification_schedule.csv  {len(schedule_df)} rows")
    print("═" * 70)

    return {
        "timing": timing_df,
        "themes": themes_df,
        "templates": templates_df,
        "schedule": schedule_df,
    }


# =============================================================================
# ENTRYPOINT
# =============================================================================
def main():
    if _is_jupyter():
        print("[INFO] Notebook/Colab detected. Using NOTEBOOK_CONFIG.")
        cfg = NOTEBOOK_CONFIG
        run(
            segments_path=cfg["segments"],
            goals_path=cfg["goals"],
            north_star_path=cfg["north_star"],
            tone_matrix_path=cfg["tone_matrix"],
            feature_map_path=cfg["feature_map"],
            output_dir=cfg["output_dir"],
            max_users_schedule=cfg.get("max_users_schedule", 2000),
        )
    else:
        import argparse

        parser = argparse.ArgumentParser(description="Task 2 — Communication & Timing Intelligence")
        parser.add_argument("--segments", default="user_segments.csv")
        parser.add_argument("--goals", default="segment_goals.csv")
        parser.add_argument("--north_star", default="company_north_star.json")
        parser.add_argument("--tone_matrix", default="allowed_tone_hook_matrix.json")
        parser.add_argument("--feature_map", default="feature_goal_map.json")
        parser.add_argument("--output_dir", default="iteration_0_before_learning")
        parser.add_argument("--max_users_schedule", type=int, default=2000)

        args, _ = parser.parse_known_args()

        run(
            segments_path=args.segments,
            goals_path=args.goals,
            north_star_path=args.north_star,
            tone_matrix_path=args.tone_matrix,
            feature_map_path=args.feature_map,
            output_dir=args.output_dir,
            max_users_schedule=args.max_users_schedule,
        )


if __name__ == "__main__":
    main()


══════════════════════════════════════════════════════════════════════
PROJECT AURORA — Task 2: Communication & Timing Intelligence
══════════════════════════════════════════════════════════════════════

[1/6] Loading inputs...
✅ Loaded 2000 users
✅ Segments found: ['Low Activity Users', 'Moderately Active Users']
✅ Loaded 0 goal rows

[2/6] Aggregating segment × lifecycle profiles...
✅ Aggregated 10 segment × lifecycle profiles

[3/6] Training LinUCB contextual bandit...
✅ LinUCB trained: 500 rounds
✅ Generated 10 timing recommendation rows
   → Saved: iteration_0_before_learning/timing_recommendations.csv

[4/6] Generating communication themes...
   OpenAI mode: ENABLED
  [ 1/10] Theme: SEG_00_PAI_HIG ... ✓
  [ 2/10] Theme: SEG_00_PAI_LOW ... ✓
  [ 3/10] Theme: SEG_00_PAI_MED ... ✓
  [ 4/10] Theme: SEG_00_TRI_HIG ... ✓
  [ 5/10] Theme: SEG_00_TRI_MED ... ✓
  [ 6/10] Theme: SEG_01_CHU_CRI ... ✓
  [ 7/10] Theme: SEG_01_CHU_HIG ... ✓
  [ 8/10] Theme: SEG_01_INA_CRI ... ✓
  [ 9/10] Them

In [ ]:
"""
Iteration 1 After Learning — Project Aurora (SpeakX)
====================================================

FIXED VERSION:
- If experiment_results.csv is missing, it auto-generates a dummy one
- Works in Colab / Jupyter / terminal
- Produces:
    iteration_1_after_learning/
      communication_themes.csv
      message_templates.csv
      timing_recommendations.csv
      user_notification_schedule.csv
      learning_delta_report.csv
"""

import os
import json
import argparse
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestRegressor

warnings.filterwarnings("ignore")

# =============================================================================
# CONFIG
# =============================================================================

TIME_WINDOWS = {
    "early_morning":  {"range": "06:00-08:59", "start": "07:00"},
    "mid_morning":    {"range": "09:00-11:59", "start": "10:00"},
    "afternoon":      {"range": "12:00-14:59", "start": "13:00"},
    "late_afternoon": {"range": "15:00-17:59", "start": "16:00"},
    "evening":        {"range": "18:00-20:59", "start": "19:00"},
    "night":          {"range": "21:00-23:59", "start": "22:00"},
}
WINDOW_NAMES = list(TIME_WINDOWS.keys())

GOOD_CTR = 15.0
GOOD_ENG = 40.0
BAD_CTR = 5.0
BAD_ENG = 20.0

GUARDRAIL_UNINSTALL_THRESHOLD = 2.0
GUARDRAIL_REDUCTION = 2

FREQ_BANDS = [
    (0.7, float("inf"), 7, 9, "high_active", "Maximum engagement"),
    (0.4, 0.7,          5, 6, "moderate_active", "Balanced approach"),
    (0.0, 0.4,          3, 4, "low_active", "Conservative, avoid fatigue"),
]

NOTEBOOK_CONFIG = {
    "segments": "user_segments.csv",
    "goals": "segment_goals.csv",
    "north_star": "company_north_star.json",
    "tone_matrix": "allowed_tone_hook_matrix.json",
    "feature_map": "feature_goal_map.json",
    "iteration0_dir": "iteration_0_before_learning",
    "experiment_results": "experiment_results.csv",
    "output_dir": "iteration_1_after_learning",
    "max_users_schedule": 2000,
    "random_state": 42,
}

# =============================================================================
# HELPERS
# =============================================================================

def _is_jupyter() -> bool:
    try:
        shell = get_ipython().__class__.__name__  # noqa
        return shell in ("ZMQInteractiveShell", "TerminalInteractiveShell", "google.colab._shell")
    except NameError:
        return False


def ensure_dir(path: str):
    Path(path).mkdir(parents=True, exist_ok=True)


def resolve_file(path: str, label: str) -> str:
    p = Path(path)
    if p.exists():
        return str(p)

    alt_roots = [Path("."), Path("/content"), Path("/mnt/data")]
    for root in alt_roots:
        candidate = root / path
        if candidate.exists():
            return str(candidate)
        candidate = root / p.name
        if candidate.exists():
            return str(candidate)

    raise FileNotFoundError(f"{label} file not found: {path}")


def file_exists_anywhere(path: str) -> bool:
    p = Path(path)
    if p.exists():
        return True
    for root in [Path("."), Path("/content"), Path("/mnt/data")]:
        if (root / path).exists() or (root / p.name).exists():
            return True
    return False


def safe_read_csv(path: str, label: str = "CSV") -> pd.DataFrame:
    path = resolve_file(path, label)
    return pd.read_csv(path)


def safe_read_json(path: str, label: str = "JSON"):
    path = resolve_file(path, label)
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def normalize_text_col(df: pd.DataFrame, col: str, lower: bool = False, default: str = ""):
    if col not in df.columns:
        df[col] = default
    df[col] = df[col].fillna(default).astype(str).str.strip()
    if lower:
        df[col] = df[col].str.lower()
    return df


def normalize_numeric_col(df: pd.DataFrame, col: str, default: float = 0.0):
    if col not in df.columns:
        df[col] = default
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(default)
    return df


def clip_pct(series: pd.Series) -> pd.Series:
    s = pd.to_numeric(series, errors="coerce").fillna(0.0)
    if len(s) and s.max() <= 1.0:
        s = s * 100.0
    return s.clip(0, 100)


def pct_from_counts(num, den):
    num = pd.to_numeric(num, errors="coerce").fillna(0.0)
    den = pd.to_numeric(den, errors="coerce").fillna(0.0).replace(0, np.nan)
    out = (num / den) * 100.0
    return out.fillna(0.0)


def clamp(x, lo, hi):
    return max(lo, min(hi, x))


def classify_performance(ctr: float, engagement_rate: float) -> str:
    if ctr > GOOD_CTR and engagement_rate > GOOD_ENG:
        return "GOOD"
    if ctr < BAD_CTR and engagement_rate < BAD_ENG:
        return "BAD"
    return "NEUTRAL"

# =============================================================================
# LOAD INPUTS
# =============================================================================

def load_segments(path: str) -> pd.DataFrame:
    df = safe_read_csv(path, "Segments")

    required_defaults = {
        "user_id": "",
        "cluster_id": 0,
        "segment_name": "Unknown Segment",
        "lifecycle_stage": "trial",
        "churn_risk": "low",
        "activeness_score": 0.3,
        "gamification_score": 0.0,
        "engagement_depth": 0.0,
        "notif_open_rate_30d": 0.2,
        "preferred_hour": 18,
        "sessions_last_7d": 0,
        "streak_current": 0,
        "motivation_score": 0.0,
        "days_since_signup": 0,
    }

    for col, default in required_defaults.items():
        if isinstance(default, str):
            normalize_text_col(df, col, lower=False, default=default)
        else:
            normalize_numeric_col(df, col, default=default)

    normalize_text_col(df, "lifecycle_stage", lower=True, default="trial")
    normalize_text_col(df, "churn_risk", lower=True, default="low")
    normalize_text_col(df, "segment_name", lower=False, default="Unknown Segment")
    df["cluster_id"] = pd.to_numeric(df["cluster_id"], errors="coerce").fillna(0).astype(int)
    df["preferred_hour"] = df["preferred_hour"].clip(0, 23)

    print(f"✅ Loaded segments: {len(df)} users")
    return df


def load_goals(path: str) -> pd.DataFrame:
    try:
        df = safe_read_csv(path, "Goals")
    except Exception:
        return pd.DataFrame()

    if df.empty:
        return pd.DataFrame()

    normalize_text_col(df, "segment_id", lower=False, default="")
    normalize_text_col(df, "segment_name", lower=False, default="")
    normalize_text_col(df, "lifecycle_stage", lower=True, default="trial")
    normalize_text_col(df, "primary_goal", lower=False, default="")
    for col in ["sub_goal_1", "sub_goal_2", "sub_goal_3", "recommended_features", "core_drive", "north_star_lever"]:
        normalize_text_col(df, col, lower=False, default="")
    return df


def load_iteration0(iteration0_dir: str):
    base = Path(resolve_file(iteration0_dir, "Iteration 0 directory"))
    files = {
        "themes": base / "communication_themes.csv",
        "templates": base / "message_templates.csv",
        "timing": base / "timing_recommendations.csv",
        "schedule": base / "user_notification_schedule.csv",
    }

    out = {}
    for key, path in files.items():
        if not path.exists():
            raise FileNotFoundError(f"Missing Iteration 0 file: {path}")
        out[key] = pd.read_csv(path)

    print("✅ Loaded Iteration 0 outputs")
    return out["themes"], out["templates"], out["timing"], out["schedule"]

# =============================================================================
# SEGMENT PROFILE AGGREGATION
# =============================================================================

def aggregate_segment_profiles(users_df: pd.DataFrame) -> pd.DataFrame:
    agg = (
        users_df.groupby(["cluster_id", "segment_name", "lifecycle_stage", "churn_risk"], dropna=False)
        .agg(
            user_count=("user_id", "count"),
            avg_activeness=("activeness_score", "mean"),
            avg_gamification=("gamification_score", "mean"),
            avg_engagement_depth=("engagement_depth", "mean"),
            avg_notif_open_rate=("notif_open_rate_30d", "mean"),
            avg_preferred_hour=("preferred_hour", "mean"),
            avg_sessions=("sessions_last_7d", "mean"),
            avg_streak=("streak_current", "mean"),
            avg_motivation=("motivation_score", "mean"),
        )
        .reset_index()
    )

    agg["segment_id"] = (
        "SEG_"
        + agg["cluster_id"].astype(int).astype(str).str.zfill(2)
        + "_"
        + agg["lifecycle_stage"].astype(str).str[:3].str.upper()
        + "_"
        + agg["churn_risk"].astype(str).str[:3].str.upper()
    )
    return agg

# =============================================================================
# AUTO-GENERATE DUMMY EXPERIMENT RESULTS IF MISSING
# =============================================================================

def hour_to_window(hour: float) -> str:
    h = int(round(float(hour))) % 24
    if 6 <= h <= 8:
        return "early_morning"
    if 9 <= h <= 11:
        return "mid_morning"
    if 12 <= h <= 14:
        return "afternoon"
    if 15 <= h <= 17:
        return "late_afternoon"
    if 18 <= h <= 20:
        return "evening"
    if 21 <= h <= 23:
        return "night"
    return "evening"


def theme_bonus(theme: str) -> float:
    mapping = {
        "accomplishment": 4.5,
        "empowerment": 4.0,
        "ownership": 3.0,
        "social influence": 2.5,
        "scarcity": 1.5,
        "epic meaning": 2.0,
        "unpredictability": 1.0,
        "loss avoidance": 2.0,
    }
    return mapping.get(str(theme).strip().lower(), 2.0)


def tone_bonus(tone: str) -> float:
    mapping = {
        "encouragement": 3.5,
        "empowerment": 3.0,
        "accomplishment": 3.0,
        "ownership": 2.5,
        "social influence": 2.0,
        "scarcity (positive)": 1.0,
        "scarcity": 1.0,
    }
    return mapping.get(str(tone).strip().lower(), 1.5)


def progression_bonus(stage: str) -> float:
    mapping = {
        "awareness": 0.8,
        "engagement": 2.0,
        "habit": 3.5,
        "mastery": 2.8,
        "retention": 2.2,
    }
    return mapping.get(str(stage).strip().lower(), 1.5)


def lifecycle_bonus(stage: str) -> float:
    mapping = {
        "trial": 1.0,
        "paid": 2.5,
        "inactive": -1.0,
        "churned": -2.0,
    }
    return mapping.get(str(stage).strip().lower(), 0.0)


def churn_penalty(churn: str) -> float:
    mapping = {
        "low": 0.0,
        "medium": -2.0,
        "high": -4.0,
        "critical": -6.0,
    }
    return mapping.get(str(churn).strip().lower(), -1.0)


def window_alignment_bonus(window: str, preferred_window: str, primary_window: str, secondary_window: str) -> float:
    bonus = 0.0
    if window == preferred_window:
        bonus += 4.5
    if window == primary_window:
        bonus += 4.0
    elif window == secondary_window:
        bonus += 2.2
    return bonus


def auto_generate_dummy_experiment_results(
    users_df: pd.DataFrame,
    templates_df: pd.DataFrame,
    timing_df: pd.DataFrame,
    schedule_df: pd.DataFrame,
    output_path: str,
    random_state: int = 42
) -> pd.DataFrame:
    rng = np.random.default_rng(random_state)
    profiles_df = aggregate_segment_profiles(users_df)

    pref_lookup = {}
    for _, r in profiles_df.iterrows():
        pref_lookup[r["segment_id"]] = {
            "avg_activeness": float(r["avg_activeness"]),
            "avg_notif_open_rate": float(r["avg_notif_open_rate"]),
            "preferred_window": hour_to_window(float(r["avg_preferred_hour"])),
            "segment_name": r["segment_name"],
            "lifecycle_stage": r["lifecycle_stage"],
            "churn_risk": r["churn_risk"],
            "user_count": int(r["user_count"]),
        }

    timing_lookup = timing_df.set_index("segment_id").to_dict("index")
    sends_by_segment = (
        schedule_df.groupby("segment_id", dropna=False)["n_notifs_today"].sum().to_dict()
        if not schedule_df.empty and "segment_id" in schedule_df.columns
        else {}
    )

    for c in ["template_id", "segment_id", "lifecycle_stage", "theme", "goal", "tone",
              "hook_type", "feature_reference", "progression_stage", "channel"]:
        normalize_text_col(templates_df, c, lower=(c in ["lifecycle_stage", "progression_stage"]), default="")

    rows = []
    windows = WINDOW_NAMES[:]

    for _, tmpl in templates_df.iterrows():
        seg_id = tmpl["segment_id"]
        seg_meta = pref_lookup.get(seg_id)
        if seg_meta is None:
            continue

        timing_meta = timing_lookup.get(seg_id, {})
        primary_window = str(timing_meta.get("primary_window", "evening")).strip().lower()
        secondary_window = str(timing_meta.get("secondary_window", "night")).strip().lower()

        seg_sends = sends_by_segment.get(seg_id, 100)
        n_templates_seg = max(1, len(templates_df[templates_df["segment_id"] == seg_id]))
        base_sends = max(40, seg_sends / n_templates_seg * 8)

        r = rng.random()
        if r < 0.65:
            chosen_window = primary_window
        elif r < 0.90:
            chosen_window = secondary_window
        else:
            chosen_window = rng.choice(windows)

        activeness = float(seg_meta["avg_activeness"])
        base_open_rate = float(seg_meta["avg_notif_open_rate"]) * 100.0
        sends = int(max(30, round(base_sends + rng.normal(0, 12))))

        ctr = (
            2.0
            + base_open_rate * 0.55
            + activeness * 12.0
            + theme_bonus(tmpl.get("theme", ""))
            + tone_bonus(tmpl.get("tone", ""))
            + progression_bonus(tmpl.get("progression_stage", ""))
            + lifecycle_bonus(tmpl.get("lifecycle_stage", ""))
            + churn_penalty(seg_meta.get("churn_risk", "low"))
            + window_alignment_bonus(chosen_window, seg_meta["preferred_window"], primary_window, secondary_window)
            + rng.normal(0, 3.5)
        )
        ctr = clamp(ctr, 1.0, 45.0)

        opens = int(round(sends * ctr / 100.0))
        opens = max(0, min(opens, sends))

        engagement_rate = (
            10.0
            + activeness * 32.0
            + progression_bonus(tmpl.get("progression_stage", "")) * 3.5
            + theme_bonus(tmpl.get("theme", "")) * 1.8
            + tone_bonus(tmpl.get("tone", "")) * 1.5
            + (4.0 if chosen_window == primary_window else 0.0)
            + (2.0 if chosen_window == secondary_window else 0.0)
            + lifecycle_bonus(tmpl.get("lifecycle_stage", "")) * 2.0
            + churn_penalty(seg_meta.get("churn_risk", "")) * 1.2
            + rng.normal(0, 6.0)
        )
        engagement_rate = clamp(engagement_rate, 5.0, 85.0)

        engagements = int(round(opens * engagement_rate / 100.0))
        engagements = max(0, min(engagements, opens))

        uninstall_rate = (
            0.5
            + max(0, 0.35 * max(3, min(9, int(round(seg_sends / max(1, seg_meta["user_count"])))) ) - 1.0)
            + (1.5 if seg_meta.get("churn_risk", "low") in ["high", "critical"] else 0.0)
            + (0.8 if chosen_window not in [primary_window, secondary_window, seg_meta["preferred_window"]] else 0.0)
            + rng.normal(0, 0.45)
        )
        uninstall_rate = clamp(uninstall_rate, 0.0, 6.5)

        rows.append({
            "template_id": tmpl["template_id"],
            "segment_id": seg_id,
            "lifecycle_stage": tmpl.get("lifecycle_stage", seg_meta["lifecycle_stage"]),
            "goal": tmpl.get("goal", "daily practice"),
            "theme": tmpl.get("theme", "Accomplishment"),
            "tone": tmpl.get("tone", "Encouragement"),
            "hook_type": tmpl.get("hook_type", "Accomplishment"),
            "feature_reference": tmpl.get("feature_reference", "AI Tutor Sia"),
            "progression_stage": tmpl.get("progression_stage", "engagement"),
            "channel": tmpl.get("channel", "push"),
            "notification_window": chosen_window,
            "total_sends": sends,
            "total_opens": opens,
            "total_engagements": engagements,
            "ctr": round(ctr, 2),
            "engagement_rate": round(engagement_rate, 2),
            "uninstall_rate": round(uninstall_rate, 2),
            "performance_status": classify_performance(ctr, engagement_rate),
        })

    result_df = pd.DataFrame(rows)

    if not result_df.empty:
        n = len(result_df)
        bad_idx = result_df.sample(max(1, n // 8), random_state=random_state).index
        result_df.loc[bad_idx, "ctr"] = result_df.loc[bad_idx, "ctr"].clip(upper=4.8)
        result_df.loc[bad_idx, "engagement_rate"] = result_df.loc[bad_idx, "engagement_rate"].clip(upper=19.0)
        result_df.loc[bad_idx, "performance_status"] = "BAD"

        remaining = result_df.index.difference(bad_idx)
        good_idx = result_df.loc[remaining].sample(max(1, n // 5), random_state=random_state + 1).index
        result_df.loc[good_idx, "ctr"] = result_df.loc[good_idx, "ctr"].clip(lower=16.5)
        result_df.loc[good_idx, "engagement_rate"] = result_df.loc[good_idx, "engagement_rate"].clip(lower=42.0)
        result_df.loc[good_idx, "performance_status"] = "GOOD"

        neutral_idx = result_df.index.difference(bad_idx.union(good_idx))
        result_df.loc[neutral_idx, "performance_status"] = result_df.loc[neutral_idx].apply(
            lambda r: classify_performance(r["ctr"], r["engagement_rate"]), axis=1
        )

        result_df["total_opens"] = (result_df["total_sends"] * result_df["ctr"] / 100.0).round().astype(int)
        result_df["total_opens"] = result_df[["total_opens", "total_sends"]].min(axis=1)
        result_df["total_engagements"] = (result_df["total_opens"] * result_df["engagement_rate"] / 100.0).round().astype(int)
        result_df["total_engagements"] = result_df[["total_engagements", "total_opens"]].min(axis=1)

    out = Path(output_path)
    out.parent.mkdir(parents=True, exist_ok=True)
    result_df.to_csv(out, index=False)

    print(f"✅ Auto-generated missing experiment results: {out}")
    print(result_df["performance_status"].value_counts(dropna=False).to_string())
    return result_df


def load_or_create_experiment_results(
    experiment_results_path: str,
    users_df: pd.DataFrame,
    templates0_df: pd.DataFrame,
    timing0_df: pd.DataFrame,
    schedule0_df: pd.DataFrame,
    random_state: int = 42
) -> pd.DataFrame:
    if file_exists_anywhere(experiment_results_path):
        df = safe_read_csv(experiment_results_path, "Experiment Results")
        print(f"✅ Loaded experiment results: {len(df)} rows")
        return df

    print("⚠️ experiment_results.csv not found. Creating dummy experiment results automatically...")
    return auto_generate_dummy_experiment_results(
        users_df=users_df,
        templates_df=templates0_df.copy(),
        timing_df=timing0_df.copy(),
        schedule_df=schedule0_df.copy(),
        output_path=experiment_results_path,
        random_state=random_state
    )

# =============================================================================
# EXPERIMENT RESULTS
# =============================================================================

def load_experiment_results(path: str) -> pd.DataFrame:
    df = safe_read_csv(path, "Experiment Results")

    normalize_text_col(df, "template_id", lower=False, default="")
    normalize_text_col(df, "segment_id", lower=False, default="")
    normalize_text_col(df, "lifecycle_stage", lower=True, default="trial")
    normalize_text_col(df, "goal", lower=False, default="")
    normalize_text_col(df, "theme", lower=False, default="")
    normalize_text_col(df, "notification_window", lower=True, default="evening")
    normalize_text_col(df, "performance_status", lower=False, default="")

    for c in ["total_sends", "total_opens", "total_engagements"]:
        normalize_numeric_col(df, c, default=0)

    if "ctr" in df.columns:
        df["ctr"] = clip_pct(df["ctr"])
    else:
        df["ctr"] = pct_from_counts(df["total_opens"], df["total_sends"])

    if "engagement_rate" in df.columns:
        df["engagement_rate"] = clip_pct(df["engagement_rate"])
    else:
        df["engagement_rate"] = pct_from_counts(df["total_engagements"], df["total_opens"].replace(0, np.nan))

    if "uninstall_rate" in df.columns:
        df["uninstall_rate"] = clip_pct(df["uninstall_rate"])
    else:
        df["uninstall_rate"] = 0.0

    if "performance_status" not in df.columns or df["performance_status"].eq("").all():
        df["performance_status"] = df.apply(
            lambda r: classify_performance(r["ctr"], r["engagement_rate"]), axis=1
        )
    else:
        df["performance_status"] = df["performance_status"].astype(str).str.strip().str.upper()
        mask = ~df["performance_status"].isin(["GOOD", "NEUTRAL", "BAD"])
        if mask.any():
            df.loc[mask, "performance_status"] = df.loc[mask].apply(
                lambda r: classify_performance(r["ctr"], r["engagement_rate"]), axis=1
            )

    df["notification_window"] = df["notification_window"].where(
        df["notification_window"].isin(WINDOW_NAMES), "evening"
    )

    print("✅ Performance mix:")
    print(df["performance_status"].value_counts(dropna=False).to_string())
    return df

# =============================================================================
# LEARNING TABLE + MODELS
# =============================================================================

def build_learning_table(experiment_df: pd.DataFrame, templates_df: pd.DataFrame, themes_df: pd.DataFrame) -> pd.DataFrame:
    tmpl = templates_df.copy()
    thm = themes_df.copy()

    normalize_text_col(tmpl, "template_id", lower=False, default="")
    normalize_text_col(tmpl, "segment_id", lower=False, default="")
    normalize_text_col(tmpl, "lifecycle_stage", lower=True, default="trial")
    normalize_text_col(tmpl, "theme", lower=False, default="")
    normalize_text_col(tmpl, "tone", lower=False, default="")
    normalize_text_col(tmpl, "hook_type", lower=False, default="")
    normalize_text_col(tmpl, "progression_stage", lower=True, default="engagement")
    normalize_text_col(tmpl, "feature_reference", lower=False, default="")
    normalize_text_col(tmpl, "channel", lower=False, default="push")
    normalize_text_col(tmpl, "goal", lower=False, default="daily practice")

    normalize_text_col(thm, "segment_id", lower=False, default="")
    for c in ["primary_theme", "secondary_theme", "primary_tone", "secondary_tone"]:
        normalize_text_col(thm, c, lower=False, default="")

    learn_df = experiment_df.merge(
        tmpl[[
            "template_id", "segment_id", "lifecycle_stage", "theme", "tone",
            "hook_type", "feature_reference", "progression_stage", "channel", "goal"
        ]],
        on="template_id",
        how="left",
        suffixes=("", "_tmpl")
    )

    for c in ["segment_id", "lifecycle_stage", "theme", "goal"]:
        alt = f"{c}_tmpl"
        if alt in learn_df.columns:
            learn_df[c] = np.where(
                learn_df[c].astype(str).str.strip().eq(""),
                learn_df[alt],
                learn_df[c]
            )

    learn_df = learn_df.merge(
        thm[["segment_id", "primary_theme", "secondary_theme", "primary_tone", "secondary_tone"]],
        on="segment_id",
        how="left"
    )

    normalize_text_col(learn_df, "segment_id", lower=False, default="")
    normalize_text_col(learn_df, "lifecycle_stage", lower=True, default="trial")
    normalize_text_col(learn_df, "theme", lower=False, default="Accomplishment")
    normalize_text_col(learn_df, "tone", lower=False, default="Encouragement")
    normalize_text_col(learn_df, "hook_type", lower=False, default="Accomplishment")
    normalize_text_col(learn_df, "feature_reference", lower=False, default="AI Tutor Sia")
    normalize_text_col(learn_df, "progression_stage", lower=True, default="engagement")
    normalize_text_col(learn_df, "channel", lower=False, default="push")
    normalize_text_col(learn_df, "notification_window", lower=True, default="evening")
    normalize_text_col(learn_df, "performance_status", lower=False, default="NEUTRAL")

    return learn_df


def train_quality_models(learn_df: pd.DataFrame, random_state: int = 42):
    df = learn_df.copy()

    feature_cols_cat = [
        "segment_id", "lifecycle_stage", "theme", "tone",
        "hook_type", "feature_reference", "progression_stage",
        "channel", "notification_window"
    ]
    feature_cols_num = ["total_sends"]

    for c in feature_cols_cat:
        normalize_text_col(df, c, lower=(c in ["lifecycle_stage", "progression_stage", "notification_window"]), default="unknown")
    for c in feature_cols_num:
        normalize_numeric_col(df, c, default=0.0)

    X = df[feature_cols_cat + feature_cols_num].copy()

    label_enc = LabelEncoder()
    y_cls = label_enc.fit_transform(df["performance_status"].astype(str).str.upper())

    pre = ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore"), feature_cols_cat),
            ("num", StandardScaler(), feature_cols_num),
        ]
    )

    cls_model = Pipeline(
        steps=[
            ("pre", pre),
            ("clf", LogisticRegression(max_iter=1000, multi_class="auto", class_weight="balanced"))
        ]
    )
    cls_model.fit(X, y_cls)

    reg_ctr = Pipeline(
        steps=[
            ("pre", pre),
            ("reg", RandomForestRegressor(
                n_estimators=250,
                random_state=random_state,
                min_samples_leaf=2
            ))
        ]
    )
    reg_ctr.fit(X, df["ctr"])

    reg_eng = Pipeline(
        steps=[
            ("pre", pre),
            ("reg", RandomForestRegressor(
                n_estimators=250,
                random_state=random_state,
                min_samples_leaf=2
            ))
        ]
    )
    reg_eng.fit(X, df["engagement_rate"])

    return {
        "label_encoder": label_enc,
        "classifier": cls_model,
        "ctr_regressor": reg_ctr,
        "eng_regressor": reg_eng,
        "feature_cols_cat": feature_cols_cat,
        "feature_cols_num": feature_cols_num,
    }

# =============================================================================
# TIMING AFTER LEARNING
# =============================================================================

def build_timing_recommendations_after_learning(profiles_df: pd.DataFrame, experiment_df: pd.DataFrame, timing0_df: pd.DataFrame) -> pd.DataFrame:
    exp = experiment_df.copy()
    normalize_text_col(exp, "segment_id", lower=False, default="")
    normalize_text_col(exp, "lifecycle_stage", lower=True, default="trial")
    normalize_text_col(exp, "notification_window", lower=True, default="evening")

    grouped = (
        exp.groupby(["segment_id", "lifecycle_stage", "notification_window"], dropna=False)
        .agg(
            sends=("total_sends", "sum"),
            avg_ctr=("ctr", "mean"),
            avg_engagement=("engagement_rate", "mean"),
            avg_uninstall=("uninstall_rate", "mean"),
        )
        .reset_index()
    )

    grouped["window_score"] = (
        0.45 * grouped["avg_ctr"] +
        0.45 * grouped["avg_engagement"] -
        0.10 * grouped["avg_uninstall"]
    ) * np.log1p(grouped["sends"])

    timing0 = timing0_df.copy()
    normalize_text_col(timing0, "segment_id", lower=False, default="")
    normalize_text_col(timing0, "lifecycle_stage", lower=True, default="trial")
    normalize_text_col(timing0, "primary_window", lower=True, default="evening")
    normalize_text_col(timing0, "secondary_window", lower=True, default="night")

    rows = []
    for _, p in profiles_df.iterrows():
        sid = p["segment_id"]
        lc = str(p["lifecycle_stage"]).lower()

        sub = grouped[(grouped["segment_id"] == sid) & (grouped["lifecycle_stage"] == lc)].copy()

        if sub.empty:
            m = timing0[timing0["segment_id"] == sid]
            if not m.empty:
                m = m.iloc[0]
                primary = m.get("primary_window", "evening")
                secondary = m.get("secondary_window", "night")
                pprob = float(m.get("primary_window_prob", 0.55))
                sprob = float(m.get("secondary_window_prob", 0.25))
            else:
                primary, secondary, pprob, sprob = "evening", "night", 0.55, 0.25

            rows.append({
                "segment_id": sid,
                "segment_name": p["segment_name"],
                "lifecycle_stage": lc,
                "churn_risk": p["churn_risk"],
                "avg_activeness": round(float(p["avg_activeness"]), 3),
                "avg_notif_open_rate": round(float(p["avg_notif_open_rate"]), 3),
                "primary_window": primary,
                "primary_window_range": TIME_WINDOWS[primary]["range"],
                "primary_window_prob": round(pprob, 4),
                "expected_ctr_primary": round(float(p["avg_notif_open_rate"]) * pprob * 100, 2),
                "secondary_window": secondary,
                "secondary_window_range": TIME_WINDOWS[secondary]["range"],
                "secondary_window_prob": round(sprob, 4),
                "expected_ctr_secondary": round(float(p["avg_notif_open_rate"]) * sprob * 100, 2),
                "learning_source": "iteration_0_fallback",
            })
            continue

        sub = sub.sort_values(["window_score", "sends"], ascending=[False, False]).reset_index(drop=True)
        ranked = sub["notification_window"].tolist()
        primary = ranked[0]
        secondary = ranked[1] if len(ranked) > 1 else ranked[0]

        scores = sub["window_score"].values.astype(float)
        e = np.exp(scores - scores.max())
        probs = e / e.sum()

        prob_map = {w: float(probs[i]) for i, w in enumerate(sub["notification_window"].tolist())}

        rows.append({
            "segment_id": sid,
            "segment_name": p["segment_name"],
            "lifecycle_stage": lc,
            "churn_risk": p["churn_risk"],
            "avg_activeness": round(float(p["avg_activeness"]), 3),
            "avg_notif_open_rate": round(float(p["avg_notif_open_rate"]), 3),
            "primary_window": primary,
            "primary_window_range": TIME_WINDOWS[primary]["range"],
            "primary_window_prob": round(prob_map.get(primary, 0.5), 4),
            "expected_ctr_primary": round(float(sub.loc[sub["notification_window"] == primary, "avg_ctr"].iloc[0]), 2),
            "secondary_window": secondary,
            "secondary_window_range": TIME_WINDOWS[secondary]["range"],
            "secondary_window_prob": round(prob_map.get(secondary, 0.25), 4),
            "expected_ctr_secondary": round(float(sub.loc[sub["notification_window"] == secondary, "avg_ctr"].iloc[0]), 2),
            "learning_source": "experiment_results",
        })

    return pd.DataFrame(rows)

# =============================================================================
# THEMES / TEMPLATES
# =============================================================================

def build_iteration1_themes(themes0_df: pd.DataFrame, experiment_df: pd.DataFrame, tone_matrix: dict) -> pd.DataFrame:
    allowed_tones = [x.get("tone", "") for x in tone_matrix.get("allowed_tones", []) if isinstance(x, dict)]
    disallowed = [x.get("tone", "") for x in tone_matrix.get("disallowed_tones", []) if isinstance(x, dict)]

    exp = experiment_df.copy()
    normalize_text_col(exp, "segment_id", lower=False, default="")
    normalize_text_col(exp, "theme", lower=False, default="")

    theme_perf = (
        exp.groupby(["segment_id", "theme"], dropna=False)
        .agg(
            avg_ctr=("ctr", "mean"),
            avg_eng=("engagement_rate", "mean"),
            avg_uninstall=("uninstall_rate", "mean"),
            sends=("total_sends", "sum"),
        )
        .reset_index()
    )

    theme_perf["score"] = (
        0.45 * theme_perf["avg_ctr"] +
        0.45 * theme_perf["avg_eng"] -
        0.10 * theme_perf["avg_uninstall"]
    ) * np.log1p(theme_perf["sends"])

    out_rows = []
    for _, row in themes0_df.iterrows():
        sid = row["segment_id"]
        sub = theme_perf[theme_perf["segment_id"] == sid].sort_values("score", ascending=False)

        new_row = row.to_dict()

        if not sub.empty:
            best_theme = sub.iloc[0]["theme"]
            second_theme = sub.iloc[1]["theme"] if len(sub) > 1 else row.get("secondary_theme", best_theme)
            new_row["primary_theme"] = best_theme or row.get("primary_theme", "Accomplishment")
            new_row["secondary_theme"] = second_theme or row.get("secondary_theme", "Empowerment")

        if new_row.get("primary_tone") not in allowed_tones:
            new_row["primary_tone"] = "Encouragement"
        if new_row.get("secondary_tone") not in allowed_tones:
            new_row["secondary_tone"] = "Empowerment"

        new_row["avoided_tones"] = disallowed[:2] if disallowed else ["Fear/Anxiety", "Shame/Guilt"]
        new_row["learning_applied"] = "yes"
        out_rows.append(new_row)

    return pd.DataFrame(out_rows)


def score_templates_for_iteration1(templates0_df: pd.DataFrame, experiment_df: pd.DataFrame, model_bundle: dict) -> pd.DataFrame:
    df = templates0_df.copy()

    for c in ["template_id", "segment_id", "lifecycle_stage", "theme", "tone",
              "hook_type", "feature_reference", "progression_stage", "channel"]:
        normalize_text_col(df, c, lower=(c in ["lifecycle_stage", "progression_stage"]), default="unknown")

    if "notification_window" not in df.columns:
        df["notification_window"] = "evening"
    normalize_text_col(df, "notification_window", lower=True, default="evening")

    if "total_sends" not in df.columns:
        hist = (
            experiment_df.groupby("template_id", dropna=False)["total_sends"]
            .sum()
            .reset_index()
            .rename(columns={"total_sends": "historical_total_sends"})
        )
        df = df.merge(hist, on="template_id", how="left")
        df["total_sends"] = df["historical_total_sends"].fillna(0)
        df.drop(columns=["historical_total_sends"], inplace=True)
    else:
        normalize_numeric_col(df, "total_sends", default=0)

    X = df[model_bundle["feature_cols_cat"] + model_bundle["feature_cols_num"]].copy()

    y_prob = model_bundle["classifier"].predict_proba(X)
    classes = model_bundle["label_encoder"].inverse_transform(np.arange(y_prob.shape[1]))
    prob_df = pd.DataFrame(y_prob, columns=[f"prob_{c}" for c in classes])

    df = pd.concat([df.reset_index(drop=True), prob_df.reset_index(drop=True)], axis=1)
    df["pred_ctr"] = model_bundle["ctr_regressor"].predict(X)
    df["pred_engagement_rate"] = model_bundle["eng_regressor"].predict(X)

    for col in ["prob_GOOD", "prob_NEUTRAL", "prob_BAD"]:
        if col not in df.columns:
            df[col] = 0.0

    df["quality_score"] = (
        1.5 * df["prob_GOOD"] +
        0.5 * df["prob_NEUTRAL"] -
        1.2 * df["prob_BAD"] +
        0.02 * df["pred_ctr"] +
        0.02 * df["pred_engagement_rate"]
    )

    return df


def build_iteration1_templates(templates0_df: pd.DataFrame, timing1_df: pd.DataFrame, scored_templates_df: pd.DataFrame) -> pd.DataFrame:
    templates = templates0_df.copy()
    scored = scored_templates_df.copy()
    timing = timing1_df.copy()

    normalize_text_col(templates, "template_id", lower=False, default="")
    normalize_text_col(templates, "segment_id", lower=False, default="")
    normalize_text_col(templates, "lifecycle_stage", lower=True, default="trial")

    normalize_text_col(scored, "template_id", lower=False, default="")
    normalize_text_col(scored, "segment_id", lower=False, default="")
    normalize_text_col(scored, "lifecycle_stage", lower=True, default="trial")

    normalize_text_col(timing, "segment_id", lower=False, default="")
    normalize_text_col(timing, "primary_window", lower=True, default="evening")
    normalize_text_col(timing, "secondary_window", lower=True, default="night")

    merged = templates.merge(
        scored[[
            "template_id", "prob_GOOD", "prob_NEUTRAL", "prob_BAD",
            "pred_ctr", "pred_engagement_rate", "quality_score"
        ]],
        on="template_id",
        how="left"
    )

    merged = merged.merge(
        timing[["segment_id", "primary_window", "secondary_window"]],
        on="segment_id",
        how="left"
    )

    merged["recommended_window"] = merged["primary_window"].fillna("evening")
    merged["backup_window"] = merged["secondary_window"].fillna("night")

    def action_from_probs(r):
        if float(r.get("prob_BAD", 0)) >= 0.50:
            return "SUPPRESS"
        if float(r.get("prob_GOOD", 0)) >= 0.50:
            return "PROMOTE"
        return "KEEP"

    merged["iteration1_action"] = merged.apply(action_from_probs, axis=1)

    promoted = merged[merged["iteration1_action"] != "SUPPRESS"].copy()
    promoted["quality_score"] = pd.to_numeric(promoted["quality_score"], errors="coerce").fillna(-999)
    promoted = promoted.sort_values(["segment_id", "lifecycle_stage", "quality_score"], ascending=[True, True, False])
    promoted = promoted.groupby(["segment_id", "lifecycle_stage"], as_index=False, group_keys=False).head(5)

    return promoted.reset_index(drop=True)

# =============================================================================
# SCHEDULE
# =============================================================================

def get_frequency_band(activeness: float, uninstall_rate_pct: float = 0.0) -> dict:
    activeness = float(activeness)
    uninstall_rate_pct = float(uninstall_rate_pct)

    for lo, hi, min_f, max_f, label, strategy in FREQ_BANDS:
        if lo <= activeness < hi or (hi == float("inf") and activeness >= lo):
            if uninstall_rate_pct > GUARDRAIL_UNINSTALL_THRESHOLD:
                min_f = max(1, min_f - GUARDRAIL_REDUCTION)
                max_f = max(1, max_f - GUARDRAIL_REDUCTION)
                strategy += " [GUARDRAIL APPLIED]"
            recommended = (min_f + max_f) // 2
            return {
                "freq_band": label,
                "min_notifs": min_f,
                "max_notifs": max_f,
                "recommended": recommended,
                "strategy": strategy,
            }

    return {"freq_band": "low_active", "min_notifs": 3, "max_notifs": 4, "recommended": 3, "strategy": "Conservative"}


def build_user_schedule(users_df: pd.DataFrame, profiles_df: pd.DataFrame, timing1_df: pd.DataFrame, templates1_df: pd.DataFrame, experiment_df: pd.DataFrame, max_users: int = 2000) -> pd.DataFrame:
    timing_lookup = timing1_df.set_index("segment_id").to_dict("index")
    template_pool = templates1_df.groupby("segment_id", dropna=False)["template_id"].apply(list).to_dict()
    channel_lookup = templates1_df.set_index("template_id")["channel"].to_dict() if "channel" in templates1_df.columns else {}

    profile_key = profiles_df.set_index(["cluster_id", "lifecycle_stage", "churn_risk"])["segment_id"].to_dict()
    uninstall_by_segment = experiment_df.groupby("segment_id", dropna=False)["uninstall_rate"].mean().to_dict()

    rng = np.random.default_rng(42)
    rows = []

    for _, u in users_df.head(max_users).iterrows():
        key = (int(u["cluster_id"]), str(u["lifecycle_stage"]).lower(), str(u["churn_risk"]).lower())
        seg_id = profile_key.get(key)
        if seg_id is None:
            continue

        tmpl_ids = template_pool.get(seg_id, [])
        if not tmpl_ids:
            continue

        timing = timing_lookup.get(seg_id, {})
        uninstall_rate = float(uninstall_by_segment.get(seg_id, 0.0))
        freq = get_frequency_band(float(u.get("activeness_score", 0.3)), uninstall_rate_pct=uninstall_rate)
        n_notifs = min(int(freq["recommended"]), 9)

        primary_window = timing.get("primary_window", "evening")
        secondary_window = timing.get("secondary_window", "night")
        primary_time = TIME_WINDOWS.get(primary_window, TIME_WINDOWS["evening"])["start"]
        secondary_time = TIME_WINDOWS.get(secondary_window, TIME_WINDOWS["night"])["start"]

        row = {
            "user_id": u["user_id"],
            "segment_id": seg_id,
            "segment_name": u["segment_name"],
            "lifecycle_stage": u["lifecycle_stage"],
            "lifecycle_day": int(pd.to_numeric(u.get("days_since_signup", 0), errors="coerce") or 0),
            "churn_risk": u["churn_risk"],
            "activeness_score": round(float(u.get("activeness_score", 0.0)), 3),
            "freq_band": freq["freq_band"],
            "n_notifs_today": n_notifs,
            "primary_window": primary_window,
            "secondary_window": secondary_window,
            "guardrail_uninstall_rate_pct": round(uninstall_rate, 3),
            "frequency_strategy": freq["strategy"],
        }

        for slot in range(1, 10):
            if slot <= n_notifs:
                tmpl = tmpl_ids[int(rng.integers(0, len(tmpl_ids)))]
                row[f"notif_{slot}_template_id"] = tmpl
                row[f"notif_{slot}_time"] = primary_time if slot % 2 == 1 else secondary_time
                row[f"notif_{slot}_channel"] = channel_lookup.get(tmpl, "push")
            else:
                row[f"notif_{slot}_template_id"] = ""
                row[f"notif_{slot}_time"] = ""
                row[f"notif_{slot}_channel"] = ""

        rows.append(row)

    return pd.DataFrame(rows)

# =============================================================================
# DELTA REPORT
# =============================================================================

def build_learning_delta_report(themes0_df: pd.DataFrame, themes1_df: pd.DataFrame, templates0_df: pd.DataFrame, templates1_df: pd.DataFrame, timing0_df: pd.DataFrame, timing1_df: pd.DataFrame, experiment_df: pd.DataFrame) -> pd.DataFrame:
    rows = []

    t0 = themes0_df.set_index("segment_id").to_dict("index")
    t1 = themes1_df.set_index("segment_id").to_dict("index")
    for sid, after in t1.items():
        before = t0.get(sid, {})
        for field in ["primary_theme", "secondary_theme", "primary_tone", "secondary_tone"]:
            before_val = before.get(field, "")
            after_val = after.get(field, "")
            if str(before_val) != str(after_val):
                rows.append({
                    "entity_type": "theme",
                    "entity_id": sid,
                    "changed_field": field,
                    "before_value": before_val,
                    "after_value": after_val,
                    "metric_trigger": "learned_performance_by_segment",
                    "explanation": "Updated theme/tone based on experiment performance signals.",
                })

    m0 = timing0_df.set_index("segment_id").to_dict("index")
    m1 = timing1_df.set_index("segment_id").to_dict("index")
    for sid, after in m1.items():
        before = m0.get(sid, {})
        for field in ["primary_window", "secondary_window"]:
            before_val = before.get(field, "")
            after_val = after.get(field, "")
            if str(before_val) != str(after_val):
                rows.append({
                    "entity_type": "timing",
                    "entity_id": sid,
                    "changed_field": field,
                    "before_value": before_val,
                    "after_value": after_val,
                    "metric_trigger": "window_score_from_experiment_results",
                    "explanation": "Timing updated using observed CTR, engagement, uninstall, and send volume.",
                })

    available_cols = [c for c in ["template_id", "iteration1_action"] if c in templates1_df.columns]
    if available_cols:
        for _, r in templates1_df[available_cols].iterrows():
            rows.append({
                "entity_type": "template",
                "entity_id": r.get("template_id", ""),
                "changed_field": "iteration1_action",
                "before_value": "iteration_0_active",
                "after_value": r.get("iteration1_action", "KEEP"),
                "metric_trigger": "predicted_quality_score",
                "explanation": f"Template assigned action {r.get('iteration1_action', 'KEEP')} for Iteration 1.",
            })

    return pd.DataFrame(rows)

# =============================================================================
# MAIN RUN
# =============================================================================

def run(
    segments_path: str,
    goals_path: str,
    north_star_path: str,
    tone_matrix_path: str,
    feature_map_path: str,
    iteration0_dir: str,
    experiment_results_path: str,
    output_dir: str,
    max_users_schedule: int = 2000,
    random_state: int = 42,
):
    ensure_dir(output_dir)

    print("\n" + "═" * 72)
    print("PROJECT AURORA — Iteration 1 After Learning")
    print("═" * 72)

    print("\n[1/8] Loading inputs...")
    users_df = load_segments(segments_path)
    _goals_df = load_goals(goals_path)
    _north_star = safe_read_json(north_star_path, "North Star JSON")
    tone_matrix = safe_read_json(tone_matrix_path, "Tone Matrix JSON")
    _feature_map = safe_read_json(feature_map_path, "Feature Map JSON")
    themes0_df, templates0_df, timing0_df, schedule0_df = load_iteration0(iteration0_dir)

    raw_experiment_df = load_or_create_experiment_results(
        experiment_results_path=experiment_results_path,
        users_df=users_df,
        templates0_df=templates0_df,
        timing0_df=timing0_df,
        schedule0_df=schedule0_df,
        random_state=random_state,
    )
    experiment_df = load_experiment_results(experiment_results_path)

    print("\n[2/8] Aggregating segment profiles...")
    profiles_df = aggregate_segment_profiles(users_df)
    print(f"✅ Profiles: {len(profiles_df)} rows")

    print("\n[3/8] Building learning table...")
    learn_df = build_learning_table(experiment_df, templates0_df, themes0_df)
    print(f"✅ Learning table: {len(learn_df)} rows")

    print("\n[4/8] Training models...")
    model_bundle = train_quality_models(learn_df, random_state=random_state)
    print("✅ Trained LogisticRegression + RandomForest models")

    print("\n[5/8] Learning timing recommendations...")
    timing1_df = build_timing_recommendations_after_learning(profiles_df, experiment_df, timing0_df)
    timing1_path = Path(output_dir) / "timing_recommendations.csv"
    timing1_df.to_csv(timing1_path, index=False)
    print(f"   → Saved: {timing1_path}")

    print("\n[6/8] Updating themes and templates...")
    themes1_df = build_iteration1_themes(themes0_df, experiment_df, tone_matrix)
    scored_templates_df = score_templates_for_iteration1(templates0_df, experiment_df, model_bundle)
    templates1_df = build_iteration1_templates(templates0_df, timing1_df, scored_templates_df)

    themes1_path = Path(output_dir) / "communication_themes.csv"
    templates1_path = Path(output_dir) / "message_templates.csv"
    themes1_df.to_csv(themes1_path, index=False)
    templates1_df.to_csv(templates1_path, index=False)
    print(f"   → Saved: {themes1_path}")
    print(f"   → Saved: {templates1_path}")

    print("\n[7/8] Building user notification schedule...")
    schedule1_df = build_user_schedule(
        users_df=users_df,
        profiles_df=profiles_df,
        timing1_df=timing1_df,
        templates1_df=templates1_df,
        experiment_df=experiment_df,
        max_users=max_users_schedule,
    )
    schedule1_path = Path(output_dir) / "user_notification_schedule.csv"
    schedule1_df.to_csv(schedule1_path, index=False)
    print(f"   → Saved: {schedule1_path}")

    print("\n[8/8] Writing learning delta report...")
    delta_df = build_learning_delta_report(
        themes0_df=themes0_df,
        themes1_df=themes1_df,
        templates0_df=templates0_df,
        templates1_df=templates1_df,
        timing0_df=timing0_df,
        timing1_df=timing1_df,
        experiment_df=experiment_df,
    )
    delta_path = Path(output_dir) / "learning_delta_report.csv"
    delta_df.to_csv(delta_path, index=False)
    print(f"   → Saved: {delta_path}")

    print("\n" + "═" * 72)
    print("ITERATION 1 COMPLETE")
    print(f"✓ timing_recommendations.csv      {len(timing1_df)} rows")
    print(f"✓ communication_themes.csv       {len(themes1_df)} rows")
    print(f"✓ message_templates.csv          {len(templates1_df)} rows")
    print(f"✓ user_notification_schedule.csv {len(schedule1_df)} rows")
    print(f"✓ learning_delta_report.csv      {len(delta_df)} rows")
    print("═" * 72)

    return {
        "timing": timing1_df,
        "themes": themes1_df,
        "templates": templates1_df,
        "schedule": schedule1_df,
        "delta": delta_df,
    }

# =============================================================================
# ENTRYPOINT
# =============================================================================

def main():
    if _is_jupyter():
        cfg = NOTEBOOK_CONFIG
        print("[INFO] Notebook/Colab detected. Using NOTEBOOK_CONFIG.")
        run(
            segments_path=cfg["segments"],
            goals_path=cfg["goals"],
            north_star_path=cfg["north_star"],
            tone_matrix_path=cfg["tone_matrix"],
            feature_map_path=cfg["feature_map"],
            iteration0_dir=cfg["iteration0_dir"],
            experiment_results_path=cfg["experiment_results"],
            output_dir=cfg["output_dir"],
            max_users_schedule=cfg["max_users_schedule"],
            random_state=cfg["random_state"],
        )
    else:
        parser = argparse.ArgumentParser(description="Iteration 1 After Learning — Project Aurora")
        parser.add_argument("--segments", default="user_segments.csv")
        parser.add_argument("--goals", default="segment_goals.csv")
        parser.add_argument("--north_star", default="company_north_star.json")
        parser.add_argument("--tone_matrix", default="allowed_tone_hook_matrix.json")
        parser.add_argument("--feature_map", default="feature_goal_map.json")
        parser.add_argument("--iteration0_dir", default="iteration_0_before_learning")
        parser.add_argument("--experiment_results", default="experiment_results.csv")
        parser.add_argument("--output_dir", default="iteration_1_after_learning")
        parser.add_argument("--max_users_schedule", type=int, default=2000)
        parser.add_argument("--random_state", type=int, default=42)

        args, _ = parser.parse_known_args()

        run(
            segments_path=args.segments,
            goals_path=args.goals,
            north_star_path=args.north_star,
            tone_matrix_path=args.tone_matrix,
            feature_map_path=args.feature_map,
            iteration0_dir=args.iteration0_dir,
            experiment_results_path=args.experiment_results,
            output_dir=args.output_dir,
            max_users_schedule=args.max_users_schedule,
            random_state=args.random_state,
        )


if __name__ == "__main__":
    main()


════════════════════════════════════════════════════════════════════════
PROJECT AURORA — Iteration 1 After Learning
════════════════════════════════════════════════════════════════════════

[1/8] Loading inputs...
✅ Loaded segments: 2000 users
✅ Loaded Iteration 0 outputs
✅ Loaded experiment results: 50 rows
✅ Performance mix:
performance_status
NEUTRAL    26
GOOD       18
BAD         6

[2/8] Aggregating segment profiles...
✅ Profiles: 10 rows

[3/8] Building learning table...
✅ Learning table: 50 rows

[4/8] Training models...
✅ Trained LogisticRegression + RandomForest models

[5/8] Learning timing recommendations...
   → Saved: iteration_1_after_learning/timing_recommendations.csv

[6/8] Updating themes and templates...
   → Saved: iteration_1_after_learning/communication_themes.csv
   → Saved: iteration_1_after_learning/message_templates.csv

[7/8] Building user notification schedule...
   → Saved: iteration_1_after_learning/user_notification_schedule.csv

[8/8] Writing learning